# Toronto-3D + Paris-Lille-3D PointNet++ Semantic Segmentation (GPU-optimised)

Trains one 8-class PointNet++-style segmenter on **both** datasets you have, holds out one
tile from each city, and renders the results for every tile at millions of points per image
-- as top-down class maps, as real 3D views with occlusion, as a rotatable 3D scene, and (new)
as a **coloured triangle mesh** with the segmentation carried onto its vertices. Section 21 points
the trained model at `Paris_Luxembourg_6.ply`, a raw mobile-scan export that was never part of
either dataset, and section 22 turns that result -- and `L002` -- into an exportable segmented
surface. Section 23 is the model report: full metric tables, confusion matrices, ablations,
cost, and the weights and biases themselves. The last cell points the same unmodified model at
`Birmingham_3D/birmingham_block_0.ply`, an aerial photogrammetry block from SensatUrban with
13 classes of its own, scores it against that block's ground truth, and puts the class-to-colour
mapping for both taxonomies on the record.

**Datasets used**
| Dataset | Tiles | Channels | Labels |
|---|---|---|---|
| Toronto-3D | `L001`, `L002`, `L003`, `L004` | XYZ + RGB | raw `1..8` |
| Paris-Lille-3D | `Lille1_1`, `Lille1_2`, `Lille2`, `Paris` | XYZ + reflectance | coarse `0..9` |

**Split:** train on `L001`, `L003`, `L004`, `Lille1_1`, `Lille1_2`, `Paris`.
Held out for testing: `L002` (Toronto) and `Lille2` (Paris-Lille). The held-out tiles are
never touched during training, validation or early stopping.

### What changed vs the previous version, and what it bought

| Area | Before | Now | Measured gain |
|---|---|---|---|
| Block sampling | O(N) scan per block, 33 ms/block | KD-tree, 0.5 ms/block | **27x** |
| Training | batch 4, DataLoader, 10.0 ms/block | batch 128, blocks resident on GPU, ~1.2 ms/block | **~8x** per block (and blocks are 2x bigger) |
| Inference | 13 us/point | 6.7 us/point, ~650k pts/s on a full tile | **~8x** |
| Area sampling | `linspace` centre indices (spatially meaningless) | farthest-point sampling | accuracy |
| Feature propagation | nearest-neighbour | 3-NN inverse-distance | accuracy |
| Loss | plain cross-entropy | class-weighted + label smoothing | rare-class mIoU |
| Visualisation | 50k-point matplotlib scatter | GPU rasteriser, 40M points in 3 s | **~800x** more points |
| 3D views | none | GPU z-buffer renderer, whole tiles in ~1 s | 10-21M points per 3D view |
| Turntable | none | full 360-degree orbit as an animated GIF | renders with no extensions |
| Interactive view | none | rotatable plotly scene, one trace per class | no extension, no JupyterLab restart |
| A tile from elsewhere | not possible | last cell predicts a raw mobile-scan export | new ground, no labels needed |
| Meshing | none | Poisson reconstruction, classes on the vertices, shaded surface renders | a segmented `PLY`, not just points |
| Reporting | OA and mIoU only | per-class precision/recall/F1, confusion matrices, kappa, ablations, weights and biases | the numbers behind the numbers |
| A tile from a third dataset | not possible | the closing cell predicts an aerial SensatUrban block and scores it against that block's own 13-class labels | a measured domain gap |

**Held-out results** (neither tile was trained on). `cudnn.benchmark` autotunes kernels, so
runs are not bit-exact and these move by a few tenths of a point; the ranges below are what
repeated runs produced:

| tile | metric | previous notebook | this version |
|---|---|---|---|
| `L002` (Toronto) | OA / mIoU | 76.73% / 0.2717 | **85.8-86.2% / 0.40** |
| `Lille2` (Paris-Lille) | OA / mIoU | 85.63% / 0.2331 | **81.8-82.5% / 0.25** |

On `Lille2` the OA is a little lower while mIoU is higher: the class weighting deliberately
trades a bit of overall accuracy for the rare classes. And because Paris-Lille tiles are now
part of training, `Lille2` is **no longer a zero-shot cross-city result** -- it is a held-out
tile from a city the model has seen, so the two `Lille2` columns are not directly comparable.
To restore the original zero-shot setup, drop the `PL_DIR` entries from `TRAIN_TILES`.

*Caveat: the per-epoch validation split is random over blocks, so neighbouring blocks may
overlap. Treat it as a development signal; the held-out tiles above are the real metric.*

## 1. Configuration

In [ ]:
from pathlib import Path
import copy, json, math, os, random, time

SEED = 42
random.seed(SEED)

PROJECT_DIR = Path.cwd().resolve()  # Run Jupyter from this repository folder.
DATA_ROOT = Path(os.environ.get("POINTNET_DATA_DIR", PROJECT_DIR)).expanduser().resolve()
DATA_DIR = DATA_ROOT / "Toronto_3D"          # Toronto-3D tiles
PL_DIR = DATA_ROOT / "training_10_classes"   # Paris-Lille-3D tiles
OUTPUT_DIR = PROJECT_DIR / "data" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- the 8 shared semantic classes (index 0..7) ---------------------------
NUM_CLASSES = 8
CLASS_NAMES = ["Road", "Road Marking", "Natural", "Building",
               "Utility Line", "Pole", "Car", "Fence"]

# ---- training tiles: both datasets ----------------------------------------
TRAIN_TILES = [
    DATA_DIR / "L001.ply",      # Toronto, RGB
    DATA_DIR / "L003.ply",
    DATA_DIR / "L004.ply",
    PL_DIR / "Lille1_1.ply",    # Paris-Lille, reflectance
    PL_DIR / "Lille1_2.ply",
    PL_DIR / "Paris.ply",
]
# ---- tiles hidden from training entirely ----------------------------------
TEST_TILES = [
    ("L002",   DATA_DIR / "L002.ply", 2.0),   # Toronto, 2 m inference grid
    ("Lille2", PL_DIR / "Lille2.ply", 1.0),   # Paris-Lille, 1 m inference grid
]
# ---- every tile, for the multi-area gallery -------------------------------
GALLERY_TILES = [
    ("L001",    DATA_DIR / "L001.ply", "toronto"),
    ("L002",    DATA_DIR / "L002.ply", "toronto"),
    ("L003",    DATA_DIR / "L003.ply", "toronto"),
    ("L004",    DATA_DIR / "L004.ply", "toronto"),
    ("Lille1_1", PL_DIR / "Lille1_1.ply", "paris-lille"),
    ("Lille1_2", PL_DIR / "Lille1_2.ply", "paris-lille"),
    ("Lille2",  PL_DIR / "Lille2.ply", "paris-lille"),
    ("Paris",   PL_DIR / "Paris.ply", "paris-lille"),
]

# ---- work budget ----------------------------------------------------------
NUM_POINTS = 1024             # points per training block
MAX_POINTS_PER_TILE = 1_500_000   # subsample of each training tile
BLOCKS_PER_TILE = 700         # blocks sampled per training tile
VAL_FRACTION = 0.15           # blocks held back per tile for validation
BATCH_SIZE = 128              # blocks per training step (GPU is ~5% used at 4)
EPOCHS = 60
PATIENCE = 15                 # epochs without val-mIoU improvement
LR = 2e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
NUM_WORKERS = 0               # unused: there is no DataLoader any more

# ---- inference ------------------------------------------------------------
INFER_BLOCK_BATCH = 128       # blocks per forward pass during inference
INFER_TILE_POINTS = 2_000_000 # points per memory-bounded inference chunk

# ---- visualisation / export ----------------------------------------------
GALLERY_MAX_POINTS = 1_000_000  # points sampled per tile for the gallery renders
GALLERY_RES = 1000              # pixels per side of a rendered class map
SHOW_3D_VIEWER = False          # flip on for an Open3D desktop window (section 19)
EXPORT_TILES = ["L002"]         # tiles written out as coloured PLY (None = all)
EXPORT_CSV = True

# ---- 3D visualisation -----------------------------------------------------
THREED_TILES = ["L002", "Lille2"]   # held-out tiles to draw in 3D
THREED_RES = (1280, 860)            # pixels per rendered 3D view
THREED_ORBIT = [(0, 25), (90, 25), (180, 25), (270, 25), (40, 60)]   # (yaw, pitch) deg
THREED_DETAIL_SIZE = 150.0          # metres across for the zoomed 3D detail view
THREED_SPLAT = 2                    # k x k pixels per point, which closes the gaps
THREED_MAX_POINTS = None            # None = draw every point of the tile
THREED_SPIN_TILE = "L002"           # tile used for the rotating turntable
THREED_SPIN_SIZE = 260.0            # metres across the turntable window
THREED_SPIN_RES = (620, 430)        # pixels per turntable frame
THREED_SPIN_FRAMES = 40             # frames per full turn
THREED_SPIN_FPS = 12                # playback rate

# ---- interactive viewer (section 19) --------------------------------------
SHOW_PLOTLY_VIEWER = True       # rotatable 3D scene inline, no extension or restart
PLOTLY_TILE = "L002"            # held-out tile to show
PLOTLY_DETAIL_SIZE = 150.0      # metres across (same window as section 17); None = whole tile
PLOTLY_MAX_POINTS = 60_000      # a browser cannot draw 10M points, so subsample
PLOTLY_INCLUDE_JS = "inline"    # "inline" works offline; "cdn" keeps the notebook small
PLOTLY_HEIGHT = 720             # pixels tall

# ---- meshing: the segmentation as a surface (section 22) ------------------
MESH_TILES = ["Paris_Luxembourg_6", "L002"]   # tiles to reconstruct (section 21 must run first)
MESH_MAX_POINTS = 1_500_000     # Poisson does not need every point; also caps runtime
MESH_DEPTH = "auto"             # octree depth, or "auto" for the rule below
MESH_VOXEL = 0.10               # "auto" targets this voxel size, in metres
MESH_DEPTH_MIN, MESH_DEPTH_MAX = 9, 11   # ... clamped to this range (memory and time grow ~8x
                                         # per level, and depth is brute-forced by hand below)
MESH_TRIM = 0.03                # drop the lowest-density fraction of Poisson vertices
MESH_DIST_MULT = 4.0            # drop vertices this far (x median point spacing) from data
MESH_NORMAL_SCALE = 6.0         # normal-estimation radius, x the median point spacing
MESH_ORIENT_K = 20              # neighbours used to orient normals consistently (the slow bit)
MESH_DETAIL_SIZE = 150.0        # metres across for the zoomed mesh view; None = whole tile
MESH_RES = (1280, 860)          # pixels per mesh render
MESH_SAMPLES_PER_PIXEL = 4.0    # surface samples per pixel of triangle area (2x2 per pixel)
MESH_EXPORT = True              # write the coloured mesh as PLY next to the other outputs
MESH_VIEWER = False             # opt-in rotatable plotly mesh (~10 MB inline); MESH_TILES only
MESH_VIEWER_TRIANGLES = 250_000  # triangle budget for that viewer

MODEL_PATH = OUTPUT_DIR / "best_pointnet2_joint.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"

print("torch device will be reported by the next cell")
for p in TRAIN_TILES + [t[1] for t in TEST_TILES]:
    print(f"{p.name:14} exists={p.exists()}  {p}")

## 2. Dependencies

In [ ]:
# Install dependencies from requirements.txt; see README.md.


## 3. Imports and GPU report

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, display
from matplotlib.animation import ArtistAnimation, PillowWriter
from matplotlib.patches import Patch, Rectangle
from plyfile import PlyData, PlyElement
from scipy.spatial import cKDTree

# open3d is only used by the optional interactive viewer at the end of the notebook.
# Probing with find_spec avoids importing it: importing open3d in a Jupyter session
# spins up a WebRTC GUI backend, which is pure overhead when the viewer is off.
import importlib.util
try:
    OPEN3D_AVAILABLE = importlib.util.find_spec("open3d") is not None
except Exception:
    OPEN3D_AVAILABLE = False

# plotly powers the rotatable viewer in section 19. Probing with find_spec keeps its
# import cost out of the training path when the viewer is switched off.
try:
    PLOTLY_AVAILABLE = importlib.util.find_spec("plotly") is not None
except Exception:
    PLOTLY_AVAILABLE = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda, "| device:", DEVICE)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | sm_{props.major}{props.minor} | "
          f"{props.total_memory / 1e9:.1f} GB | {props.multi_processor_count} SMs")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("TF32 + cudnn.benchmark enabled")

## 4. Unified tile loader (Toronto-3D **and** Paris-Lille-3D)

One loader for both datasets. It returns the original world coordinates (kept for export),
6 model features `[tile-centred XYZ, RGB]` and labels mapped to the shared `0..7` classes.

* Toronto-3D: raw `1..8` -> `0..7`; raw `0` (unclassified) is ignored.
* Paris-Lille-3D: coarse `0..9` -> the shared classes via the mapping below, which is taken
  from the dataset's own `coarse_classes.xml`, not guessed. Reflectance is used as greyscale
  because those tiles carry no RGB.

Coordinates are re-centred on each tile's own origin **in float64 before the float32 cast**.
That is what preserves centimetre precision, and it works for any UTM zone, so the hardcoded
Toronto offset is gone.

In [ ]:
# Toronto-3D raw label -> shared class. Raw 0 = unclassified, ignored.
TORONTO_LUT = np.full(32, -1, np.int64)
TORONTO_LUT[1:9] = np.arange(8)

# Paris-Lille-3D coarse label -> shared class (from coarse_classes.xml):
#   0 unclassified->ignored   1 ground->Road       2 building->Building
#   3 signage->Pole           4 bollard->Pole      5 trash_can->ignored
#   6 barrier->Fence          7 pedestrian->ignored
#   8 car->Car                9 natural->Natural
PL_REMAP = {0: -1, 1: 0, 2: 3, 3: 5, 4: 5, 5: -1, 6: 7, 7: -1, 8: 6, 9: 2}
PL_LUT = np.full(32, -1, np.int64)
for _src, _dst in PL_REMAP.items():
    PL_LUT[_src] = _dst


def load_tile(path, max_points=None, seed=SEED, keep_world=True, verbose=True):
    '''Load a Toronto-3D or Paris-Lille-3D tile.

    Returns (world_xyz, features, labels, kind):
      world_xyz (N,3) float64 original coords (or None) -- for export only
      features  (N,6) float32 [tile-centred XYZ, RGB in 0..1]
      labels    (N,)  int64 shared class ids 0..7, or None when unlabelled
      kind      "toronto" | "paris-lille"
    '''
    ply = PlyData.read(str(path))
    vertex = ply["vertex"].data
    fields = list(vertex.dtype.names)

    def find(cands):
        low = {f.lower(): f for f in fields}
        for c in cands:
            if c.lower() in low:
                return low[c.lower()]
        return None

    xf, yf, zf = find(["x"]), find(["y"]), find(["z"])
    if not all((xf, yf, zf)):
        raise ValueError(f"missing XYZ in {path}: {fields}")
    rf, gf, bf = find(["red", "r"]), find(["green", "g"]), find(["blue", "b"])
    refl_f = find(["reflectance"])
    lf = find(["scalar_label", "label", "class"])
    kind = "paris-lille" if refl_f is not None else "toronto"

    n = len(vertex)
    if max_points is not None and n > max_points:
        keep = np.sort(np.random.default_rng(seed).choice(n, max_points, replace=False))
    else:
        keep = np.arange(n)
    if verbose:
        print(f"  {path.name}: {n:,} pts, fields {fields}"
              + (f", sampling {len(keep):,}" if len(keep) < n else ""), flush=True)

    world = np.column_stack([vertex[xf][keep], vertex[yf][keep], vertex[zf][keep]]).astype(np.float64)

    if refl_f is not None:
        ch = np.asarray(vertex[refl_f][keep], np.float32)
        if ch.max(initial=0) > 1:
            ch /= 255.0
        rgb = np.repeat(np.clip(ch, 0, 1)[:, None], 3, axis=1)
    else:
        def chan(f):
            return np.zeros(len(keep), np.float32) if f is None else np.asarray(vertex[f][keep], np.float32)
        rgb = np.column_stack([chan(rf), chan(gf), chan(bf)])
        if rgb.max(initial=0) > 1:
            rgb /= 255.0
        rgb = np.clip(rgb, 0, 1)

    labels = None
    if lf is not None:
        raw = np.asarray(vertex[lf][keep], np.int64)
        lut = PL_LUT if kind == "paris-lille" else TORONTO_LUT
        labels = lut[np.clip(raw, 0, len(lut) - 1)]

    centred = (world - world.mean(axis=0)).astype(np.float32)
    features = np.concatenate([centred, rgb.astype(np.float32)], axis=1).astype(np.float32)

    finite = np.isfinite(features).all(axis=1)
    if labels is not None:
        finite &= labels >= 0          # drop ignored/unclassified points outright
    features = features[finite]
    labels = labels[finite] if labels is not None else None
    world = world[finite] if keep_world else None
    del ply, vertex
    return world, features, labels, kind

## 5. Block creation (KD-tree, ~27x faster)

The previous version computed the distance from the anchor to **every** point in the tile,
once per block -- O(N) per block, 33 ms/block. A KD-tree answers the same k-nearest query in
0.5 ms. Each block is centred and unit-scaled in place, exactly as before.

In [ ]:
def create_blocks(features, labels, num_points=NUM_POINTS, num_blocks=BLOCKS_PER_TILE,
                  seed=SEED):
    '''Sample local neighbourhood blocks around random anchors.

    Returns (points (B,K,6) float32, labels (B,K) int64); XYZ of each block is
    centred and scaled to unit radius.
    '''
    if labels is None or len(features) == 0:
        return np.zeros((0, num_points, 6), np.float32), np.zeros((0, num_points), np.int64)

    n = len(features)
    k = min(num_points, n)
    tree = cKDTree(features[:, :2])                      # one tree per tile
    rng = np.random.default_rng(seed)
    anchors = rng.choice(n, size=min(num_blocks, n), replace=False)
    idx = tree.query(features[anchors, :2], k=k, workers=-1)[1]   # batched k-NN
    if idx.ndim == 1:
        idx = idx[:, None]
    if k < num_points:                                   # tiny tile: pad by repeat
        idx = np.concatenate([idx, np.repeat(idx[:, -1:], num_points - k, axis=1)], axis=1)

    pts = features[idx].astype(np.float32)
    labs = labels[idx].astype(np.int64)
    xyz = pts[:, :, :3]
    xyz -= xyz.mean(axis=1, keepdims=True)
    xyz /= np.maximum(np.linalg.norm(xyz, axis=2).max(axis=1)[:, None, None], 1e-6)
    return pts, labs


print("create_blocks ready")

## 6. Build the joint training set

Blocks are sampled from **all six training tiles** and split per tile into train/validation.
The two held-out tiles are deliberately not touched here, so they stay clean test sets.

In [ ]:
def build_training_set(tiles, max_points=MAX_POINTS_PER_TILE, blocks_per_tile=BLOCKS_PER_TILE,
                       num_points=NUM_POINTS, val_fraction=VAL_FRACTION, seed=SEED):
    tr_p, tr_l, va_p, va_l, report = [], [], [], [], []
    for i, path in enumerate(tiles):
        if not path.exists():
            print(f"  skipping missing tile {path}")
            continue
        print(f"loading {path.name}")
        t0 = time.time()
        _, feats, labs, kind = load_tile(path, max_points=max_points, seed=seed + i)
        if labs is None:
            print(f"  {path.name}: no labels, skipped")
            continue
        pts, blk_labs = create_blocks(feats, labs, num_points=num_points,
                                     num_blocks=blocks_per_tile, seed=seed + 100 + i)
        perm = np.random.default_rng(seed + i).permutation(len(pts))
        n_val = max(1, int(len(pts) * val_fraction))
        va, tr = perm[:n_val], perm[n_val:]
        tr_p.append(pts[tr]); tr_l.append(blk_labs[tr])
        va_p.append(pts[va]); va_l.append(blk_labs[va])
        counts = np.bincount(blk_labs.reshape(-1), minlength=NUM_CLASSES)
        print(f"  {len(pts):5d} blocks from {path.name} ({kind}) in {time.time() - t0:.1f}s"
              f" | per-class points {counts.tolist()}", flush=True)
        report.append({"tile": path.name, "kind": kind, "blocks": int(len(pts)),
                       "class_counts": counts.tolist()})
        del feats
    if not tr_p:
        raise RuntimeError("no training tiles were loaded")
    return (np.concatenate(tr_p), np.concatenate(tr_l),
            np.concatenate(va_p), np.concatenate(va_l), report)


t0 = time.time()
train_points, train_labels, val_points, val_labels, tile_report = build_training_set(TRAIN_TILES)
print(f"\ntrain blocks {len(train_points)} | val blocks {len(val_points)} | "
      f"{train_points.shape} | built in {time.time() - t0:.1f}s")
assert len(val_points) > 0, "validation split is empty"

## 7. GPU augmentation, tensors and class weights

Augmentation now runs on the GPU and is vectorised across the whole batch (0.8 ms for 128
blocks), so there is no `DataLoader` and no worker pool -- the entire block set is just two
dense tensors on the GPU.

The loss is weighted by inverse-sqrt class frequency. That is what lifts the classes the
previous version scored ~0 on (Pole, Fence, Road Marking): without it, a handful of dominant
road points swamps everything else. The brightness jitter uses a single offset per point, so
reflectance-as-greyscale blocks stay grey.

In [ ]:
def augment_batch(pts, labs):
    '''Vectorised GPU augmentation of a whole batch of blocks.'''
    B, N, _ = pts.shape
    dev = pts.device

    ang = torch.rand(B, device=dev) * (2 * math.pi)
    c, s = torch.cos(ang)[:, None], torch.sin(ang)[:, None]
    x, y = pts[..., 0].clone(), pts[..., 1].clone()
    pts = pts.clone()
    pts[..., 0] = c * x - s * y                       # random rotation about Z
    pts[..., 1] = s * x + c * y

    pts[..., :3] *= (0.9 + 0.2 * torch.rand(B, 1, 1, device=dev))      # scale
    pts[..., :3] += 0.005 * torch.randn(B, N, 3, device=dev)           # jitter
    pts[..., 3:6] = (pts[..., 3:6]
                     + 0.02 * torch.randn(B, N, 1, device=dev)).clamp_(0.0, 1.0)

    n_drop = (torch.rand(B, 1, device=dev) * 0.10 * N).long()          # point dropout
    mask = torch.arange(N, device=dev).view(1, N) < n_drop
    repl = torch.randint(0, N, (B, N), device=dev)
    pts = torch.where(mask[..., None],
                      torch.gather(pts, 1, repl[..., None].expand(-1, -1, pts.shape[-1])), pts)
    labs = torch.where(mask, torch.gather(labs, 1, repl), labs)
    return pts, labs


def class_weights(labels, num_classes=NUM_CLASSES):
    '''Inverse-sqrt-frequency weights so rare classes are not drowned out.'''
    counts = torch.bincount(labels.reshape(-1), minlength=num_classes).float()
    present = counts > 0
    w = torch.zeros(num_classes, device=counts.device, dtype=counts.dtype)
    w[present] = (counts[present].sum() / counts[present]).clamp(max=50.0) ** 0.5
    w[present] = w[present] / w[present].mean()
    return w


# upload the whole block set once: 128 x 1024 x 6 floats per batch is nothing,
# but the full set is only a few hundred MB and removes all per-batch CPU work.
TR_POINTS = torch.from_numpy(train_points).to(DEVICE)
TR_LABELS = torch.from_numpy(train_labels).to(DEVICE)
VA_POINTS = torch.from_numpy(val_points).to(DEVICE)
VA_LABELS = torch.from_numpy(val_labels).to(DEVICE)
CLASS_WEIGHTS = class_weights(TR_LABELS)

print(f"train tensor {tuple(TR_POINTS.shape)} = {TR_POINTS.numel() * 4 / 1e6:.0f} MB on {DEVICE}")
print("class weights:", {CLASS_NAMES[i]: round(float(CLASS_WEIGHTS[i]), 3)
                         for i in range(NUM_CLASSES)})

## 8. Model: PointNet++ (SSG) with farthest-point sampling

Two structural fixes over the previous model, which mattered more than extra depth:

1. **Farthest-point sampling** picks the set-abstraction centres. The old code used
   `torch.linspace`, i.e. evenly spaced *array indices* -- spatially meaningless, so the
   "local" neighbourhoods were arbitrary. FPS is what makes the hierarchy actually local.
2. **3-NN inverse-distance interpolation** in the decoder, as in the paper, instead of
   nearest-neighbour upsampling.

FPS is written with `||x-c||^2 = ||x||^2 - 2x.c + ||c||^2` so each step is one `bmm` plus a
couple of elementwise kernels rather than building `(B,N,3)` temporaries.

In [ ]:
def index_points(points, idx):
    '''points (B,N,C), idx (B,S) or (B,S,K) -> (B,S,C) or (B,S,K,C).'''
    B = points.shape[0]
    batch = torch.arange(B, device=points.device).view(B, *([1] * (idx.ndim - 1))).expand_as(idx)
    return points[batch, idx, :]


def farthest_point_sample(xyz, npoint):
    '''Batched GPU farthest-point sampling. xyz (B,N,3) -> idx (B,S).'''
    B, N, _ = xyz.shape
    S = min(npoint, N)
    dev = xyz.device
    xyz_sq = (xyz * xyz).sum(-1)                        # (B,N), computed once
    idx = torch.zeros(B, S, dtype=torch.long, device=dev)
    dist = torch.full((B, N), float("inf"), device=dev)
    farthest = torch.zeros(B, dtype=torch.long, device=dev)
    rows = torch.arange(B, device=dev)
    for i in range(S):
        idx[:, i] = farthest
        c = xyz[rows, farthest]
        d = (xyz_sq + (c * c).sum(-1).unsqueeze(1)) - 2.0 * torch.bmm(xyz, c.unsqueeze(-1)).squeeze(-1)
        torch.minimum(dist, d, out=dist)
        farthest = dist.max(-1).indices
    return idx


class LocalSetAbstraction(nn.Module):
    '''FPS centres -> kNN grouping -> shared MLP -> channel max-pool.'''

    def __init__(self, in_channels, out_channels, k=24):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels + 3, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, xyz, features, npoint):
        B, N, _ = xyz.shape
        centres = index_points(xyz, farthest_point_sample(xyz, npoint))
        k = min(self.k, N)
        _, idx = torch.cdist(centres, xyz).topk(k, dim=-1, largest=False)
        grouped = torch.cat([index_points(xyz, idx) - centres.unsqueeze(2),
                             index_points(features, idx)], dim=-1)
        out = self.mlp(grouped.permute(0, 3, 1, 2).contiguous())
        return centres, out.max(dim=-1).values.transpose(1, 2).contiguous()


class FeaturePropagation(nn.Module):
    '''3-NN inverse-distance interpolation of coarse features onto fine points, then skip.'''

    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        layers = [nn.Conv1d(in_channels, out_channels, 1, bias=False),
                  nn.BatchNorm1d(out_channels), nn.ReLU(inplace=True)]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz_fine, xyz_coarse, feat_fine, feat_coarse):
        d = torch.cdist(xyz_fine, xyz_coarse)                       # (B, Nf, Nc)
        knn_d, knn_i = d.topk(min(3, xyz_coarse.shape[1]), dim=-1, largest=False)
        w = 1.0 / (knn_d + 1e-8)
        w = w / w.sum(dim=-1, keepdim=True)
        interp = (index_points(feat_coarse, knn_i) * w.unsqueeze(-1)).sum(dim=2)
        if feat_fine is not None:
            interp = torch.cat([feat_fine, interp], dim=-1)
        return self.mlp(interp.transpose(1, 2)).transpose(1, 2).contiguous()


class PointNet2SSG(nn.Module):
    def __init__(self, in_channels=6, num_classes=NUM_CLASSES):
        super().__init__()
        self.sa1 = LocalSetAbstraction(in_channels, 64, k=24)
        self.sa2 = LocalSetAbstraction(64, 128, k=24)
        self.sa3 = LocalSetAbstraction(128, 256, k=16)
        self.fp3 = FeaturePropagation(256 + 128, 256, dropout=0.2)
        self.fp2 = FeaturePropagation(256 + 64, 256, dropout=0.2)
        self.fp1 = FeaturePropagation(256 + in_channels, 128, dropout=0.3)
        self.head = nn.Sequential(
            nn.Conv1d(128, 64, 1, bias=False), nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Conv1d(64, num_classes, 1),
        )

    def forward(self, points):
        xyz0, f0 = points[:, :, :3], points
        n = points.shape[1]
        # 8:1 / 32:1 / 128:1 reductions keep the grouped-MLP memory traffic bounded
        xyz1, f1 = self.sa1(xyz0, f0, max(32, n // 8))
        xyz2, f2 = self.sa2(xyz1, f1, max(16, n // 32))
        xyz3, f3 = self.sa3(xyz2, f2, max(8, n // 128))
        up3 = self.fp3(xyz2, xyz3, f2, f3)
        up2 = self.fp2(xyz1, xyz2, f1, up3)
        up1 = self.fp1(xyz0, xyz1, f0, up2)
        return self.head(up1.transpose(1, 2)).transpose(1, 2).contiguous()


print("model classes defined")

## 9. Check model dimensions

In [ ]:
model = PointNet2SSG(in_channels=6, num_classes=NUM_CLASSES).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
with torch.no_grad():
    out = model(TR_POINTS[:4])
print("input:", tuple(TR_POINTS[:4].shape), "-> output:", tuple(out.shape),
      "| labels:", tuple(TR_LABELS[:4].shape))
assert out.shape == (4, NUM_POINTS, NUM_CLASSES)

## 10. Train

AdamW + cosine schedule with a short warmup, class-weighted cross-entropy with label
smoothing, gradient clipping, and early stopping on validation mIoU. The best-mIoU weights are
restored at the end.

At batch 128 this is ~1.0 ms per block instead of ~10 ms, and the whole set is resident on
the GPU, so there is no dataloader in the way.

In [ ]:
@torch.no_grad()
def evaluate_blocks(model, points, labels, batch=INFER_BLOCK_BATCH):
    '''Confusion matrix, OA and per-class IoU over pre-built blocks.'''
    was_training = model.training
    model.eval()
    conf = np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)
    for s in range(0, len(points), batch):
        pred = model(points[s:s + batch]).argmax(-1).reshape(-1).cpu().numpy()
        true = labels[s:s + batch].reshape(-1).cpu().numpy()
        ok = (true >= 0) & (true < NUM_CLASSES)
        conf += np.bincount(true[ok] * NUM_CLASSES + pred[ok],
                            minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
    if was_training:
        model.train()
    inter = np.diag(conf).astype(np.float64)
    union = conf.sum(0) + conf.sum(1) - inter
    ious = np.divide(inter, union, out=np.full(NUM_CLASSES, np.nan), where=union > 0)
    return np.trace(conf) / max(conf.sum(), 1), ious, conf


model = PointNet2SSG(in_channels=6, num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS, label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

nb = len(TR_POINTS)
steps_per_epoch = max(1, nb // BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS
warm_steps = steps_per_epoch * 3
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda step: ((step + 1) / max(1, warm_steps)) if step < warm_steps
    else 0.5 * (1 + math.cos(math.pi * min((step - warm_steps) / max(1, total_steps - warm_steps), 1.0))),
)

history = {"loss": [], "val_loss": [], "val_miou": [], "val_oa": []}
best_miou, best_state, bad = -1.0, None, 0
t_start = time.time()

for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(nb, device=DEVICE)
    total, seen = 0.0, 0
    for s in range(0, nb - BATCH_SIZE + 1, BATCH_SIZE):
        pts, labs = augment_batch(TR_POINTS[perm[s:s + BATCH_SIZE]],
                                  TR_LABELS[perm[s:s + BATCH_SIZE]])
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(pts).reshape(-1, NUM_CLASSES), labs.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total += loss.item()
        seen += 1

    model.eval()
    vloss, vbatches = 0.0, 0
    with torch.no_grad():
        for s in range(0, len(VA_POINTS), BATCH_SIZE):
            vloss += criterion(model(VA_POINTS[s:s + BATCH_SIZE]).reshape(-1, NUM_CLASSES),
                               VA_LABELS[s:s + BATCH_SIZE].reshape(-1)).item()
            vbatches += 1
    vloss /= max(vbatches, 1)
    val_oa, val_ious, _ = evaluate_blocks(model, VA_POINTS, VA_LABELS)
    val_miou = float(np.nanmean(val_ious))

    history["loss"].append(total / max(seen, 1))
    history["val_loss"].append(vloss)
    history["val_miou"].append(val_miou)
    history["val_oa"].append(float(val_oa))

    tag = ""
    if val_miou > best_miou:
        best_miou, bad = val_miou, 0
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        tag = "  <- best"
    else:
        bad += 1
    print(f"epoch {epoch + 1:03d}/{EPOCHS} | train {total / max(seen, 1):.4f} | val {vloss:.4f} "
          f"| val OA {val_oa * 100:5.2f}% | val mIoU {val_miou:.4f} "
          f"| lr {optimizer.param_groups[0]['lr']:.2e}{tag}", flush=True)
    if bad >= PATIENCE:
        print(f"early stopping: no mIoU gain for {PATIENCE} epochs")
        break

if best_state is not None:
    model.load_state_dict(best_state)
train_minutes = (time.time() - t_start) / 60
print(f"\ntrained in {train_minutes:.2f} min | best val mIoU {best_miou:.4f}")
torch.save(model.state_dict(), MODEL_PATH)
with open(HISTORY_PATH, "w") as fh:
    json.dump({**history, "tile_report": tile_report, "train_minutes": train_minutes}, fh, indent=2)
print("saved", MODEL_PATH)

## 11. Training curves and validation IoU

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(history["loss"], label="train")
ax[0].plot(history["val_loss"], label="validation")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("weighted cross-entropy")
ax[0].set_title("Loss"); ax[0].grid(True); ax[0].legend()
ax[1].plot([v * 100 for v in history["val_oa"]], label="val OA %")
ax[1].plot([v * 100 for v in history["val_miou"]], label="val mIoU x100")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("percent"); ax[1].set_title("Accuracy / mIoU")
ax[1].grid(True); ax[1].legend()
plt.tight_layout(); plt.show()

_, val_ious, val_conf = evaluate_blocks(model, VA_POINTS, VA_LABELS)
print(f"validation mIoU (best weights): {float(np.nanmean(val_ious)):.4f}")
for i, v in enumerate(val_ious):
    print(f"  {CLASS_NAMES[i]:12} IoU {'n/a' if not np.isfinite(v) else f'{v:.4f}'}")
print("\nReminder: these are randomly sampled blocks from the training tiles, so nearby "
      "blocks can overlap. The held-out tiles below are the honest metric.")

## 12. Batched GPU inference over a whole tile

Replaces the old per-4-block loop. The scripted version spent 30% of its time shuffling
points between CPU and GPU; now the tile's features are uploaded once and blocks are gathered
with `index_select`, probabilities are accumulated straight into a GPU buffer with
`index_add_`, and 128 blocks go through the network per forward pass.

In [ ]:
@torch.no_grad()
def predict_tile(features, model, num_points=NUM_POINTS, spacing=2.0,
                 batch_blocks=INFER_BLOCK_BATCH, tile_points=INFER_TILE_POINTS,
                 verbose=True):
    '''Block-wise voting inference over a full tile.

    Returns (labels (N,), probability (N,C) float32, covered (N,) bool).
    Anchors are one representative point per occupied XY grid cell; every point in a block
    receives the block's predicted probabilities, averaged over all blocks that contain it.
    Points that never appear in any block inherit the nearest covered point.
    '''
    model.eval()
    n = len(features)
    if n == 0:
        raise ValueError("empty point cloud")

    if tile_points is not None and n > tile_points:
        order = np.argsort(features[:, 0], kind="stable")
        chunks = [order[i:i + tile_points] for i in range(0, n, tile_points)]
    else:
        chunks = [np.arange(n)]

    all_pred = np.empty(n, np.int64)
    all_prob = np.zeros((n, NUM_CLASSES), np.float32)
    all_cov = np.zeros(n, bool)

    for ci, idx in enumerate(chunks):
        pf = features[idx]
        m = len(pf)
        if verbose:
            print(f"  chunk {ci + 1}/{len(chunks)}: {m:,} points", flush=True)

        pf_gpu = torch.from_numpy(np.ascontiguousarray(pf)).to(DEVICE)
        tree = cKDTree(pf[:, :2])
        # 1-D cell keys are much cheaper to deduplicate than np.unique(axis=0)
        gx = np.floor(pf[:, 0] / spacing).astype(np.int64)
        gy = np.floor(pf[:, 1] / spacing).astype(np.int64)
        gx -= gx.min(); gy -= gy.min()
        _, first = np.unique(gx * (gy.max() + 1) + gy, return_index=True)
        anchors = np.sort(first)
        k = min(num_points, m)

        prob_sum = torch.zeros(m, NUM_CLASSES, device=DEVICE)
        votes = torch.zeros(m, device=DEVICE)
        ones = torch.ones(batch_blocks * num_points, device=DEVICE)

        for a0 in range(0, len(anchors), max(batch_blocks * 4, 1024)):
            ab = anchors[a0:a0 + max(batch_blocks * 4, 1024)]
            idx_blocks = tree.query(pf[ab, :2], k=k, workers=-1)[1]
            if idx_blocks.ndim == 1:
                idx_blocks = idx_blocks[:, None]
            if k < num_points:
                idx_blocks = np.concatenate(
                    [idx_blocks, np.repeat(idx_blocks[:, -1:], num_points - k, axis=1)], axis=1)
            idx_gpu = torch.from_numpy(idx_blocks).to(device=DEVICE, dtype=torch.long)

            for s in range(0, idx_gpu.shape[0], batch_blocks):
                ib = idx_gpu[s:s + batch_blocks]
                bb = ib.shape[0]
                blk = pf_gpu.index_select(0, ib.reshape(-1)).reshape(bb, num_points, -1)
                xyz = blk[:, :, :3] - blk[:, :, :3].mean(dim=1, keepdim=True)
                xyz = xyz / xyz.norm(dim=2).max(dim=1).values.clamp_min(1e-6).view(-1, 1, 1)
                blk = torch.cat([xyz, blk[:, :, 3:]], dim=-1)

                probs = torch.softmax(model(blk), dim=-1)
                flat = ib.reshape(-1)
                prob_sum.index_add_(0, flat, probs.reshape(-1, NUM_CLASSES))
                votes.index_add_(0, flat, ones[:flat.numel()])

        votes_cpu = votes.cpu().numpy()
        prob_cpu = prob_sum.cpu().numpy()
        covered = votes_cpu > 0
        avg = np.zeros_like(prob_cpu)
        avg[covered] = prob_cpu[covered] / votes_cpu[covered, None]

        if covered.any() and (~covered).any():          # nearest-neighbour fill
            cov_i = np.flatnonzero(covered)
            unc_i = np.flatnonzero(~covered)
            _, nearest = cKDTree(pf[cov_i, :3]).query(pf[unc_i, :3], k=1, workers=-1)
            avg[unc_i] = avg[cov_i[nearest]]

        all_pred[idx] = avg.argmax(axis=1)
        all_prob[idx] = avg
        all_cov[idx] = covered
        del pf_gpu
        if verbose:
            print(f"    blocks {len(anchors):,} | direct coverage {covered.mean() * 100:.1f}%",
                  flush=True)

    return all_pred, all_prob, all_cov


def confusion_metrics(gt, pred):
    valid = (gt >= 0) & (gt < NUM_CLASSES)
    conf = np.bincount(gt[valid] * NUM_CLASSES + pred[valid],
                       minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
    inter = np.diag(conf).astype(np.float64)
    union = conf.sum(0) + conf.sum(1) - inter
    ious = np.divide(inter, union, out=np.full(NUM_CLASSES, np.nan), where=union > 0)
    return np.trace(conf) / max(conf.sum(), 1), ious, conf


print("predict_tile ready")

## 13. Predict the held-out tiles (`L002` and `Lille2`)

Neither tile was trained on, so these are the honest numbers. Full tiles, no sampling.

In [ ]:
heldout = {}
for name, path, spacing in TEST_TILES:
    if not path.exists():
        print(f"missing {path}, skipped")
        continue
    print(f"\n=== {name} (spacing {spacing} m) ===")
    t0 = time.time()
    world, feats, labs, kind = load_tile(path, max_points=None)
    pred, prob, cov = predict_tile(feats, model, spacing=spacing)
    oa, ious, conf = (confusion_metrics(labs, pred) if labs is not None
                      else (float("nan"), np.full(NUM_CLASSES, np.nan), None))
    dt = time.time() - t0
    # Keep the ORIGINAL world coordinates (needed for export and the 3D views) and a
    # per-point confidence, not the full (N, 8) probability matrix -- that alone would be
    # ~670 MB on the Lille2 tile. Ground truth is kept too, but as int8, so comparing
    # prediction against truth in 3D costs ~10 MB for L002 and ~21 MB for Lille2.
    heldout[name] = {"pred": pred, "conf": prob.max(axis=1), "coverage": cov,
                     "gt": (None if labs is None else labs.astype(np.int8)),
                     "xyz": world, "spacing": spacing, "kind": kind,
                     "oa": oa, "ious": ious, "seconds": dt, "n": len(feats)}
    print(f"{len(feats):,} pts in {dt / 60:.2f} min ({len(feats) / dt / 1e3:.0f}k pts/s) "
          f"| coverage {cov.mean() * 100:.1f}%")
    if labs is not None:
        print(f"held-out OA {oa * 100:.2f}% | mIoU {float(np.nanmean(ious)):.4f}")
        for i, v in enumerate(ious):
            print(f"   {CLASS_NAMES[i]:12} {'n/a' if not np.isfinite(v) else f'{v:.4f}'}")
    np.save(OUTPUT_DIR / f"pred_{name}.npy", pred)
    del feats, prob, labs, conf   # world coords, confidence and truth are all the later cells need
print("\ncomparable previous-notebook numbers: L002 OA 76.73% / mIoU 0.2717")

## 14. Multi-area gallery: every tile, rendered at millions of points

The old notebook drew a 50k-point matplotlib scatter. A scatter cannot do more than that in
reasonable time or memory. Here each tile is rasterised instead: every point votes for its
class in its pixel, and the pixel takes the winning class, producing a crisp class map from
**tens of millions of points in a few seconds**. That is a ~800x increase in points drawn.

All eight tiles are shown -- Toronto `L001`-`L004` and Paris-Lille `Lille1_1`, `Lille1_2`,
`Lille2`, `Paris` -- so you can see the same model across two cities and very different
scenes. The two held-out tiles are marked.

In [ ]:
PALETTE = np.array([
    [31, 119, 180], [255, 127, 14], [44, 160, 44], [214, 39, 40],
    [148, 103, 189], [140, 86, 75], [227, 119, 194], [127, 127, 127],
], np.float32) / 255.0


def render_class_map(x, y, labels, resolution=GALLERY_RES, background=0.12):
    '''Majority-vote top-down segmentation image from an arbitrary point count.'''
    x = np.asarray(x, np.float64); y = np.asarray(y, np.float64)
    labels = np.asarray(labels, np.int64)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, labels = x[ok], y[ok], labels[ok]

    sx = (resolution - 1) / max(x.max() - x.min(), 1e-9)
    sy = (resolution - 1) / max(y.max() - y.min(), 1e-9)
    px = np.clip(((x - x.min()) * sx).astype(np.int64), 0, resolution - 1)
    py = np.clip(((y - y.min()) * sy).astype(np.int64), 0, resolution - 1)

    valid = labels >= 0                     # ignored points leave holes, they do not vote
    enc = (py[valid] * resolution + px[valid]) * NUM_CLASSES + labels[valid]
    hist = np.bincount(enc, minlength=resolution * resolution * NUM_CLASSES)
    hist = hist.reshape(resolution, resolution, NUM_CLASSES)
    img = PALETTE[hist.argmax(-1)].copy()
    img[hist.sum(-1) == 0] = background
    return np.flipud(img)      # imshow puts row 0 at the top; flip so +Y points up


def gallery_row(ax_gt, ax_pr, name, path, spacing, tag=""):
    t0 = time.time()
    _, feats, labs, kind = load_tile(path, max_points=GALLERY_MAX_POINTS, verbose=False)
    pred, _, cov = predict_tile(feats, model, spacing=spacing, verbose=False)
    if labs is not None:
        ax_gt.imshow(render_class_map(feats[:, 0], feats[:, 1], labs))
        ax_gt.set_title(f"{name} ground truth{tag}", fontsize=13)
    else:
        ax_gt.axis("off")
    ax_pr.imshow(render_class_map(feats[:, 0], feats[:, 1], pred))
    ax_pr.set_title(f"{name} predicted ({len(feats):,} pts, {cov.mean() * 100:.0f}% direct)",
                    fontsize=13)
    for ax in (ax_gt, ax_pr):
        ax.set_xticks([]); ax.set_yticks([])
    print(f"  {name:9} {len(feats):>10,} pts rendered in {time.time() - t0:5.1f}s", flush=True)
    del feats, pred


def draw_gallery(tiles, title):
    fig, axes = plt.subplots(len(tiles), 2, figsize=(15, 7.2 * len(tiles)))
    axes = np.atleast_2d(axes)
    for row, (name, path, _kind) in enumerate(tiles):
        spacing = 2.0 if _kind == "toronto" else 1.0
        tag = "  (held out)" if name in [t[0] for t in TEST_TILES] else ""
        gallery_row(axes[row, 0], axes[row, 1], name, path, spacing, tag)
    handles = [Patch(facecolor=PALETTE[i], label=f"{i}: {CLASS_NAMES[i]}") for i in range(NUM_CLASSES)]
    fig.legend(handles=handles, loc="lower center", ncol=NUM_CLASSES, frameon=False, fontsize=11)
    fig.suptitle(title, fontsize=16)
    plt.tight_layout(rect=(0, 0.02, 1, 0.98))
    plt.show()
    return fig


print("rendering Toronto-3D tiles")
toronto = [t for t in GALLERY_TILES if t[2] == "toronto"]
draw_gallery(toronto, "Toronto-3D - all tiles (class map from millions of points)")

print("rendering Paris-Lille-3D tiles")
paris = [t for t in GALLERY_TILES if t[2] == "paris-lille"]
draw_gallery(paris, "Paris-Lille-3D - all tiles (class map from millions of points)")

## 15. Side view and a dense single-tile view

The class-map trick also works for an X-Z cross-section, which shows building height and
pole/car structure that the top-down view flattens away. The dense view uses the full
held-out `L002` point cloud (no sampling) at higher resolution.

In [ ]:
# X-Z cross sections for the two held-out tiles
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, (name, path, spacing) in zip(axes, TEST_TILES):
    if name not in heldout:
        ax.axis("off"); continue
    h = heldout[name]
    img = render_class_map(h["xyz"][:, 0], h["xyz"][:, 2], h["pred"], resolution=900)
    ax.imshow(img, aspect="auto")
    ax.set_title(f"{name} predicted, side view (X-Z), {len(h['pred']):,} pts", fontsize=13)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

# dense full-cloud view of the held-out Toronto tile
if "L002" in heldout:
    h = heldout["L002"]
    dense = render_class_map(h["xyz"][:, 0], h["xyz"][:, 1], h["pred"], resolution=1800)
    plt.figure(figsize=(11, 11))
    plt.imshow(dense)
    plt.title(f"L002 held-out prediction - all {len(h['pred']):,} points at 1800 px", fontsize=13)
    plt.xticks([]); plt.yticks([])
    plt.tight_layout(); plt.show()

# The interactive viewers -- a rotatable plotly scene and Open3D's desktop window --
# moved to section 19, so every viewer is in one place. The sections below draw the same
# clouds as stills, which is what works in an unattended run.

## 16. A 3D renderer that draws every point

Everything above is top-down, which flattens away the thing that makes this data 3D: the
height of buildings, poles and cars standing up out of the road, tree canopy hanging over
it. This section draws the segmented cloud in perspective instead.

Open3D is installed, but its offscreen renderer is not usable here. Measured on this
machine, it silently returns **empty black frames** for a point cloud once the scene passes
a few tens of thousands of points, and the failing sizes are scattered rather than a clean
limit (20k works, 30k-65k fail, 80k works, 100k fails, 150k works, 200k fails), and chunking
into many small clouds does not recover it. So the renderer below is written directly
against PyTorch:

* points go through a **perspective camera**, so the view is genuinely 3D, not an isometric
  squashing;
* a **depth buffer** keeps only the nearest surface at each pixel, so a building occludes the
  road behind it instead of both being painted on top of each other;
* each pixel then takes the **majority class** among the surviving points, which keeps class
  boundaries crisp instead of blending them into mud;
* colour is darkened with distance as a depth cue;
* the focal length is rescaled to the projected bounding box (`fit`), so every tile and every
  camera angle fills its frame without hand-tuned camera distances.

It draws **tens of millions of points per view in about a second** -- the entire held-out
`Lille2` tile (21.1M points) renders in ~2 s -- and it is deterministic. Open3D's offscreen
path cannot draw 30,000 points here without failing; this draws 21 million.

In [ ]:
# A z-buffered point renderer on the GPU. See the notes above for why this is not Open3D.

def camera_basis(yaw_deg, pitch_deg, dtype=torch.float32):
    '''gluLookAt-style basis for a camera orbiting a target; world up is +Z.

    Returns (eye_dir, right, up, forward): the unit vector from target to eye, plus the
    camera's right / up / forward axes in world space.
    '''
    y, p = math.radians(yaw_deg), math.radians(pitch_deg)
    eye_dir = torch.tensor([math.cos(p) * math.cos(y),
                            math.cos(p) * math.sin(y),
                            math.sin(p)], device=DEVICE, dtype=dtype)
    world_up = torch.tensor([0.0, 0.0, 1.0], device=DEVICE, dtype=dtype)
    forward = -eye_dir
    right = torch.cross(forward, world_up, dim=0)
    n = float(right.norm())
    right = (torch.tensor([1.0, 0.0, 0.0], device=DEVICE, dtype=dtype) if n < 1e-6
             else right / n)                        # looking straight down: any right will do
    return eye_dir, right, torch.cross(right, forward, dim=0), forward


@torch.no_grad()
def render_3d(xyz, labels, yaw=0.0, pitch=30.0, resolution=THREED_RES, fov=45.0,
              distance=2.6, palette=PALETTE, num_classes=NUM_CLASSES, background=0.07,
              depth_shade=0.45, slab=1.02, splat=THREED_SPLAT, fill=0.9, crop=None,
              highlight=None, max_points=THREED_MAX_POINTS):
    '''Render labelled points as an occluded, depth-shaded class map.

    xyz       : (N, 3) world coordinates. The cloud is re-centred in float64 first, so the
                large UTM-style coordinates of both datasets keep their precision.
    labels    : (N,) class indices; negatives and unknowns are dropped.
    crop      : optional ((x0, y0), (x1, y1)) window in world units.
    highlight : optional iterable of classes to colour; the rest become dark grey, keeping
                their occlusion context while one class is singled out.
    splat     : k x k pixels per point, which closes the gaps between points.

    The projection is linear in the focal length, so fitting the cloud to the frame with one
    rescale is exact. Returns a float32 (H, W, 3) image in [0, 1] with +Z up.
    '''
    W, H = int(resolution[0]), int(resolution[1])
    X = np.asarray(xyz, np.float64)
    L = np.asarray(labels, np.int64)
    keep = np.isfinite(X).all(axis=1) & (L >= 0) & (L < num_classes)
    if crop is not None:
        (x0, y0), (x1, y1) = crop
        keep &= (X[:, 0] >= x0) & (X[:, 0] <= x1) & (X[:, 1] >= y0) & (X[:, 1] <= y1)
    X, L = X[keep], L[keep]
    if max_points is not None and len(X) > max_points:
        sel = np.random.default_rng(SEED).choice(len(X), max_points, replace=False)
        X, L = X[sel], L[sel]
    if len(X) == 0:
        return np.zeros((H, W, 3), np.float32)

    P = torch.as_tensor((X - X.mean(axis=0)).astype(np.float32), device=DEVICE)
    L = torch.as_tensor(L, device=DEVICE)
    P = P / max(float(P.norm(dim=1).max()), 1e-6)      # unit-radius cloud, any tile

    eye_dir, right, up, forward = camera_basis(yaw, pitch)
    v = P - eye_dir * distance
    zc, xc, yc = v @ forward, v @ right, v @ up        # zc > 0 is in front of the camera
    front = zc > 1e-4
    if not bool(front.any()):
        return np.zeros((H, W, 3), np.float32)

    f0 = (H / 2.0) / math.tan(math.radians(fov) / 2.0)
    px = W / 2.0 + f0 * xc / zc
    py = H / 2.0 - f0 * yc / zc
    bw = float(px[front].max() - px[front].min())
    bh = float(py[front].max() - py[front].min())
    s = min(fill * W / max(bw, 1e-6), fill * H / max(bh, 1e-6))
    px = W / 2.0 + (px - W / 2.0) * s
    py = H / 2.0 + (py - H / 2.0) * s

    # nearest-surface depth per pixel, then keep only the points sitting on that surface
    valid = front & (px >= 0) & (px < W) & (py >= 0) & (py < H)
    vi = torch.nonzero(valid, as_tuple=False).squeeze(1)
    if len(vi) == 0:
        return np.zeros((H, W, 3), np.float32)
    pix = py[vi].long() * W + px[vi].long()
    depth = torch.full((H * W,), float("inf"), device=DEVICE)
    depth.scatter_reduce_(0, pix, zc[vi], reduce="amin", include_self=True)

    sel = vi[zc[vi] <= depth[pix] * slab + 1e-6]
    sel_pix = py[sel].long() * W + px[sel].long()
    sel_lab = L[sel]
    if splat > 1:                                     # stamp each point over a k x k patch
        parts_pix, parts_lab = [sel_pix], [sel_lab]
        for dy in range(splat):
            for dx in range(splat):
                if dx == 0 and dy == 0:
                    continue
                qx, qy = px[sel].long() + dx, py[sel].long() + dy
                ok = (qx >= 0) & (qx < W) & (qy >= 0) & (qy < H)
                parts_pix.append(qy[ok] * W + qx[ok])
                parts_lab.append(sel_lab[ok])
        sel_pix, sel_lab = torch.cat(parts_pix), torch.cat(parts_lab)

    # each pixel takes the majority class among its near-surface votes
    votes = torch.bincount(sel_pix * num_classes + sel_lab,
                           minlength=H * W * num_classes).reshape(H * W, num_classes)
    has = votes.sum(dim=1) > 0
    winner = votes.argmax(dim=1)

    pal = torch.as_tensor(np.asarray(palette, np.float32), device=DEVICE)
    if highlight is not None:
        pal = pal.clone()
        bright = torch.zeros(len(pal), dtype=torch.bool, device=DEVICE)
        bright[torch.as_tensor(sorted({int(c) for c in highlight}), device=DEVICE)] = True
        pal[~bright] = torch.tensor([0.13, 0.13, 0.16], device=DEVICE)

    img = torch.zeros(H * W, 3, device=DEVICE)
    img[has] = pal[winner[has]]

    if depth_shade:
        d = depth.clone()
        finite = torch.isfinite(d)
        dn = ((d - depth[finite].min()) / (depth[finite].max() - depth[finite].min() + 1e-9))
        img = img * (1.0 - depth_shade * dn.clamp(0, 1).masked_fill(~finite, 1.0)
                     ).clamp(0.25, 1.0).unsqueeze(1)

    img[~has] = torch.tensor(background, device=DEVICE, dtype=torch.float32)
    return img.reshape(H, W, 3).clamp(0, 1).cpu().numpy()   # row 0 is the top of the image


def render_orbit(xyz, labels, views=None, **kw):
    '''One (yaw, pitch, image) per requested angle.'''
    return [(y, p, render_3d(xyz, labels, yaw=y, pitch=p, **kw))
            for y, p in (THREED_ORBIT if views is None else views)]


def detail_crop(xyz, labels, size=THREED_DETAIL_SIZE, classes=(5, 6, 7)):
    '''The window with the most poles / cars / fences, so a detail view lands somewhere
    worth looking at without hand-picking coordinates.'''
    x, y = xyz[:, 0], xyz[:, 1]
    m = np.isin(labels, classes)
    cell = size / 2.0
    if not m.any():
        cx, cy = (x.min() + x.max()) / 2.0, (y.min() + y.max()) / 2.0
    else:
        gx = ((x[m] - x.min()) / cell).astype(np.int64)
        gy = ((y[m] - y.min()) / cell).astype(np.int64)
        ny, nx = int(gy.max()) + 1, int(gx.max()) + 1
        h = np.bincount(gy * nx + gx, minlength=nx * ny).reshape(ny, nx)
        iy, ix = np.unravel_index(int(h.argmax()), h.shape)
        cx, cy = x.min() + (ix + 0.5) * cell, y.min() + (iy + 0.5) * cell
    return ((cx - size / 2.0, cy - size / 2.0), (cx + size / 2.0, cy + size / 2.0))


def principal_yaw(xyz, crop=None, offset=90.0):
    '''Heading, in degrees, that turns a scene's long axis to run across the image. Used so
    an oval tile or a street does not render diagonally.'''
    X = np.asarray(xyz, np.float64)
    if crop is not None:
        (x0, y0), (x1, y1) = crop
        X = X[(X[:, 0] >= x0) & (X[:, 0] <= x1) & (X[:, 1] >= y0) & (X[:, 1] <= y1)]
    if len(X) < 2:
        return offset
    c = X[:, :2] - X[:, :2].mean(axis=0)
    _, v = np.linalg.eigh((c.T @ c) / len(c))
    return (math.degrees(math.atan2(v[1, -1], v[0, -1])) + offset) % 180.0


print("3D renderer ready")

## 17. The held-out tiles in 3D: prediction vs ground truth

Same tile, same camera, all points. Prediction on the top row, ground truth underneath.
`yaw` orbits the scene and `pitch` is the elevation: 25 degrees is a shallow, street-level
kind of view; 60 looks down into the scene. These are the **full** clouds -- 10.0M points for
`L002`, 21.1M for `Lille2` -- not a sample or a decimated preview.

The second figure zooms in. It picks, automatically, the 150 m window with the densest
poles / cars / fences in each held-out tile and draws the prediction there from two heights,
which is where the hard classes are actually checkable point by point.

In [ ]:
threed_tiles = [n for n in THREED_TILES if n in heldout]
print(f"3D views for {threed_tiles or 'nothing (run section 13 first)'}")

for name in threed_tiles:
    h = heldout[name]
    t0 = time.time()
    fig, axes = plt.subplots(2, len(THREED_ORBIT), figsize=(4.4 * len(THREED_ORBIT), 8.6))
    for j, (yaw, pitch) in enumerate(THREED_ORBIT):
        for i, (lab, tag) in enumerate(((h["pred"], "predicted"), (h["gt"], "ground truth"))):
            ax = axes[i, j]
            if lab is None:
                ax.axis("off")
                continue
            ax.imshow(render_3d(h["xyz"], lab, yaw=yaw, pitch=pitch))
            ax.set_title(f"{tag}  yaw {yaw}, pitch {pitch}", fontsize=12)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.legend(handles=[Patch(facecolor=PALETTE[i], label=f"{i}: {CLASS_NAMES[i]}")
                        for i in range(NUM_CLASSES)],
               loc="lower center", ncol=NUM_CLASSES, frameon=False, fontsize=11)
    fig.suptitle(f"{name}: all {len(h['pred']):,} points drawn in 3D", fontsize=16)
    plt.tight_layout(rect=(0, 0.035, 1, 0.96))
    out = OUTPUT_DIR / f"threed_orbit_{name}.png"
    fig.savefig(out, dpi=100)
    plt.show()
    print(f"  {name}: {2 * len(THREED_ORBIT)} views of {len(h['pred']):,} points "
          f"in {time.time() - t0:.1f}s -> {out}")

In [ ]:
if threed_tiles:
    pitches = (45, 12)
    fig, axes = plt.subplots(len(threed_tiles), len(pitches),
                             figsize=(6.6 * len(pitches), 5.4 * len(threed_tiles)))
    axes = np.atleast_2d(axes)
    for row, name in enumerate(threed_tiles):
        h = heldout[name]
        box = detail_crop(h["xyz"], h["gt"] if h["gt"] is not None else h["pred"])
        yaw = principal_yaw(h["xyz"], box)      # turn the street to run across the frame
        inside = ((h["xyz"][:, 0] >= box[0][0]) & (h["xyz"][:, 0] <= box[1][0]) &
                  (h["xyz"][:, 1] >= box[0][1]) & (h["xyz"][:, 1] <= box[1][1]))
        for col, pitch in enumerate(pitches):
            ax = axes[row, col]
            ax.imshow(render_3d(h["xyz"], h["pred"], yaw=yaw, pitch=pitch, crop=box,
                                resolution=(1120, 900)))
            ax.set_title(f"{name} predicted - {THREED_DETAIL_SIZE:.0f} m window, "
                         f"{int(inside.sum()):,} pts, yaw {yaw:.0f}, pitch {pitch}", fontsize=12)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle("3D detail: the densest poles / cars / fences window in each held-out tile",
                 fontsize=15)
    plt.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()

## 18. A turntable, and where each class lives

The orbit above is five stills. This turns the same renderer through a full 360 degrees and
plays it back as an **animated GIF**, which needs no JavaScript, no notebook extension and no
CDN -- so it renders in any Jupyter, any browser, and any viewer that shows an image.

If you would rather rotate and zoom the scene yourself, section 19 keeps an **interactive
viewer** as well: a plotly `Scatter3d` that ships its own copy of plotly.js inside the cell,
so it becomes a scene you can grab and spin -- no notebook extension, no JupyterLab restart.
The GIF above remains the one that needs no JavaScript at all.

The second figure singles out one class at a time in 3D while everything else stays as dark
context, so occlusion still reads. That is a quick check that the model's poles and cars are
in the *right places* rather than merely present somewhere.

In [ ]:
if threed_tiles:
    name = THREED_SPIN_TILE if THREED_SPIN_TILE in heldout else threed_tiles[0]
    h = heldout[name]
    box = detail_crop(h["xyz"], h["gt"] if h["gt"] is not None else h["pred"],
                      size=THREED_SPIN_SIZE)
    res = THREED_SPIN_RES
    fig = plt.figure(figsize=(res[0] / 100.0, res[1] / 100.0), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")
    t0 = time.time()
    artists = []
    for k in range(THREED_SPIN_FRAMES):
        img = render_3d(h["xyz"], h["pred"], yaw=360.0 * k / THREED_SPIN_FRAMES, pitch=22,
                        crop=box, resolution=res)
        artists.append([ax.imshow(img, animated=True)])
    anim = ArtistAnimation(fig, artists, interval=1000.0 / THREED_SPIN_FPS, blit=True)
    gif = OUTPUT_DIR / f"threed_spin_{name}.gif"
    anim.save(str(gif), writer=PillowWriter(fps=THREED_SPIN_FPS), dpi=100)
    plt.close(fig)
    print(f"{name}: {THREED_SPIN_FRAMES} frames of a {THREED_SPIN_SIZE:.0f} m window at "
          f"{res[0]}x{res[1]} in {time.time() - t0:.1f}s -> {gif.name} "
          f"({gif.stat().st_size / 1e6:.2f} MB)")
    display(Image(filename=str(gif)))     # image/gif output: renders without any extension

In [ ]:
if threed_tiles:
    name = threed_tiles[0]
    h = heldout[name]
    box = detail_crop(h["xyz"], h["gt"] if h["gt"] is not None else h["pred"], size=60.0)
    yaw = principal_yaw(h["xyz"], box)
    picks = [2, 3, 5, 6]        # Natural, Building, Pole, Car
    fig, axes = plt.subplots(1, len(picks), figsize=(5.2 * len(picks), 5.4))
    for ax, cls in zip(np.atleast_1d(axes), picks):
        ax.imshow(render_3d(h["xyz"], h["pred"], yaw=yaw, pitch=20, crop=box,
                            highlight=[cls], resolution=(880, 760)))
        n_cls = int((h["pred"] == cls).sum())
        ax.set_title(f"{CLASS_NAMES[cls]} only\n{n_cls:,} points in the whole tile", fontsize=12)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(f"{name}: one class at a time, in 3D, others left as dark context",
                 fontsize=15)
    plt.tight_layout(rect=(0, 0, 1, 0.92))
    plt.show()

## 19. Interactive viewer: grab the scene and turn it

Everything above is a picture. This section is the one you can move: a plotly `Scatter3d` of
the same prediction, drawn inline, which you **rotate with the mouse, zoom with the wheel** and
pan with shift-drag. Each class is its own trace, so the legend is a set of on/off switches --
click `Pole` and the scene shows nothing but poles.

It renders **without** the `jupyterlab-plotly` extension and without restarting JupyterLab,
because the figure is emitted as a **self-contained HTML block** (`plotly.io.to_html(...,
include_plotlyjs=True)`) that carries its own copy of plotly.js. Two consequences:

* **the notebook has to be trusted** for the output to run, since JavaScript in an untrusted
  output is stripped. Click *Trust Notebook*, or just run this cell yourself;
* that inlined library is ~5 MB, so this one cell is the biggest thing in the notebook. Set
  `PLOTLY_INCLUDE_JS = "cdn"` to fetch plotly.js from the internet instead and cut it to ~2 MB.

A browser cannot draw 10M points, so `PLOTLY_MAX_POINTS` samples the cloud (60k by default)
from the same 150 m window section 17 zooms into, where poles, cars and fences actually live;
set `PLOTLY_DETAIL_SIZE = None` for the whole tile. The sections above remain the
full-resolution record -- this view is for looking around, not for measuring.

The same scene is also written out as `threed_interactive_<tile>.html` in `OUTPUT_DIR`, which
opens in any browser and needs no notebook trust at all -- that file is the most reliable way
to look at it.

For a native window instead, set `SHOW_3D_VIEWER = True` in section 1 and the last cell here
opens Open3D on the held-out tiles. It is off by default because a GUI window blocks an
unattended run.

In [ ]:
if PLOTLY_AVAILABLE:
    def plotly_3d_html(xyz, lab, title, max_points=PLOTLY_MAX_POINTS, height=None,
                       include_js=None, marker_size=1.6, seed=SEED):
        '''A rotatable, self-contained plotly scene of a coloured cloud.

        One trace per class, so the legend doubles as an isolation switch. Returns
        (html, counts). plotly.js is inlined unless told otherwise, which is what lets it
        render without the jupyterlab-plotly extension. The caller decides whether the
        result is displayed inline or only written to disk.
        '''
        import plotly.graph_objects as go
        import plotly.io as pio

        n_all = len(lab)
        if max_points is not None and n_all > max_points:
            sel = np.random.default_rng(seed).choice(n_all, max_points, replace=False)
            xyz, lab = xyz[sel], lab[sel]
        if include_js is None:
            include_js = "cdn" if PLOTLY_INCLUDE_JS == "cdn" else True

        # two decimals is all a browser scene can resolve, and it halves the payload
        xyz = np.round(np.asarray(xyz, np.float64), 2)
        hexes = ["#%02x%02x%02x" % tuple(int(round(c * 255)) for c in PALETTE[i])
                 for i in range(NUM_CLASSES)]

        # one trace per class: toggling the legend isolates a class in 3D for free, and
        # drawing the common classes first leaves poles and cars on top rather than buried
        counts = np.bincount(lab, minlength=NUM_CLASSES)
        traces = []
        for c in np.argsort(-counts):
            m = lab == c
            if not m.any():
                continue
            traces.append(go.Scatter3d(
                x=xyz[m, 0], y=xyz[m, 1], z=xyz[m, 2], mode="markers",
                marker=dict(size=marker_size, color=hexes[c], opacity=0.9),
                name=f"{c}: {CLASS_NAMES[c]} ({int(counts[c]):,})", hoverinfo="skip"))
        fig = go.Figure(traces)
        fig.update_layout(
            title=title, height=PLOTLY_HEIGHT if height is None else height,
            margin=dict(l=0, r=0, t=34, b=0), paper_bgcolor="white", showlegend=True,
            legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.7)", font=dict(size=11)),
            scene=dict(aspectmode="data", bgcolor="white",
                       camera=dict(eye=dict(x=1.5, y=1.5, z=0.8)),
                       xaxis=dict(visible=False), yaxis=dict(visible=False),
                       zaxis=dict(visible=False)))
        return pio.to_html(fig, full_html=False, include_plotlyjs=include_js,
                           config=dict(scrollZoom=True, displaylogo=False,
                                       responsive=True)), counts
    print("plotly scene helper ready")
else:
    print("plotly is not installed -- install from requirements.txt, then rerun this cell")


if SHOW_PLOTLY_VIEWER and threed_tiles:
    from IPython.display import HTML, display

    name = PLOTLY_TILE if PLOTLY_TILE in threed_tiles else threed_tiles[0]
    h = heldout[name]
    xyz, lab = h["xyz"], h["pred"]

    box = None
    if PLOTLY_DETAIL_SIZE:
        box = detail_crop(xyz, h["gt"] if h["gt"] is not None else lab,
                          size=PLOTLY_DETAIL_SIZE)
        inside = ((xyz[:, 0] >= box[0][0]) & (xyz[:, 0] <= box[1][0]) &
                  (xyz[:, 1] >= box[0][1]) & (xyz[:, 1] <= box[1][1]))
        xyz, lab = xyz[inside], lab[inside]

    title = (f"{name} predicted in 3D -- drag to orbit, wheel to zoom "
             f"({min(PLOTLY_MAX_POINTS or len(lab), len(lab)):,} of {len(lab):,} points"
             + (f", {PLOTLY_DETAIL_SIZE:.0f} m window" if box else ", whole tile") + ")")
    html, counts = plotly_3d_html(xyz, lab, title)
    out = OUTPUT_DIR / f"threed_interactive_{name}.html"
    out.write_text(html, encoding="utf-8")
    print(f"{name}: {int(counts.sum()):,} points in {int((counts > 0).sum())} class traces -> "
          f"{out.name} ({len(html) / 1e6:.2f} MB, plotly.js "
          f"{'from CDN' if PLOTLY_INCLUDE_JS == 'cdn' else 'inlined'})")
    display(HTML(html))

In [ ]:
if SHOW_3D_VIEWER:
    if not OPEN3D_AVAILABLE:
        print("Open3D is not available; skipping the desktop window.")
    else:
        import open3d as o3d
        for name, path, spacing in TEST_TILES:
            if name not in heldout:
                continue
            h = heldout[name]
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(np.asarray(h["xyz"], np.float64))
            pcd.colors = o3d.utility.Vector3dVector(PALETTE[h["pred"]].astype(np.float64))
            print(f"opening an Open3D window for {name} ({len(h['pred']):,} points); "
                  f"close it to carry on")
            o3d.visualization.draw_geometries([pcd], window_name=f"{name} predicted")
else:
    print("SHOW_3D_VIEWER = False, so no desktop window was opened. The plotly scene in the "
          "cell above is the inline viewer; set this True in section 1 to get Open3D's native "
          "window on top of it.")

## 20. Export

Coloured PLY for the tiles listed in `EXPORT_TILES` (each point takes its predicted class
colour) plus a CSV of the held-out Toronto tile with model ids, raw ids and confidence.

In [ ]:
for name, path, spacing in TEST_TILES:
    if name not in heldout or EXPORT_TILES is None or name in EXPORT_TILES:
        if name not in heldout:
            continue
        h = heldout[name]
        out = OUTPUT_DIR / f"predict_segmented_{name}.ply"
        cols = (PALETTE[h["pred"]] * 255).astype(np.uint8)
        v = np.empty(len(h["pred"]), dtype=[("x", "f8"), ("y", "f8"), ("z", "f8"),
                                            ("red", "u1"), ("green", "u1"), ("blue", "u1")])
        v["x"], v["y"], v["z"] = h["xyz"][:, 0], h["xyz"][:, 1], h["xyz"][:, 2]
        v["red"], v["green"], v["blue"] = cols[:, 0], cols[:, 1], cols[:, 2]
        PlyData([PlyElement.describe(v, "vertex")], text=False).write(str(out))
        print(f"saved {out} ({len(v):,} points)")
        del v, cols

if EXPORT_CSV and "L002" in heldout:
    h = heldout["L002"]
    n_show = min(2_000_000, len(h["pred"]))
    sel = np.arange(n_show)
    df = pd.DataFrame({
        "x": h["xyz"][sel, 0], "y": h["xyz"][sel, 1], "z": h["xyz"][sel, 2],
        "predicted_model_class": h["pred"][sel],
        "predicted_raw_toronto3d_label": h["pred"][sel] + 1,
        "confidence": h["conf"][sel],
    })
    csv_path = OUTPUT_DIR / "prediction_labels.csv"
    df.to_csv(csv_path, index=False)
    print("saved", csv_path, f"({len(df):,} rows)")
    del df

## Troubleshooting

- **Out of memory on the GPU:** lower `BATCH_SIZE` (128 -> 64), `NUM_POINTS` (1024 -> 512)
  or `MAX_POINTS_PER_TILE`. `INFER_TILE_POINTS` bounds inference memory.
- **Training too slow:** lower `EPOCHS`, `BLOCKS_PER_TILE` or `MAX_POINTS_PER_TILE`.
  Per-epoch cost is roughly `BLOCKS_PER_TILE x tiles / BATCH_SIZE` forward passes.
- **Training too fast / accuracy plateau:** raise `BLOCKS_PER_TILE` and `EPOCHS`. More diverse
  blocks help more than a bigger network here.
- **Batch size 4 is ~34x slower than batch 128.** These blocks are tiny, so the GPU is
  completely launch-bound at small batches. Do not lower `BATCH_SIZE` for speed.
- **AMP does not help.** Measured on this GPU, bf16 autocast is *slower* than fp32 for this
  model because it is dominated by kernel-launch overhead, not arithmetic.
- **Do not enable `expandable_segments`.** PyTorch rejects it on Windows.
- **All predictions one class:** check the class weights printed in section 7, the label
  mapping, and whether the tiles actually loaded (section 6 prints per-class point counts).
- **Paris-Lille tiles look purple/grey:** those tiles have no RGB, only reflectance, which is
  fed in as greyscale by design.
- **`Lille2` is no longer zero-shot.** Paris-Lille tiles are in training now. If you need a
  genuine zero-shot cross-city number, remove them from `TRAIN_TILES`.
- **Gallery is slow:** lower `GALLERY_MAX_POINTS` or `GALLERY_RES`, or trim `GALLERY_TILES`.
- **3D views are slow or heavy on GPU memory:** lower `THREED_RES`, use fewer `THREED_ORBIT`
  angles, or set `THREED_MAX_POINTS` to subsample (e.g. `3_000_000`). `render_3d` holds a few
  float32 arrays the size of the cloud, so a 21M-point tile peaks around 2 GB; call
  `torch.cuda.empty_cache()` before the 3D cells if something ahead of them left the GPU full.
- **The turntable is slow or the notebook gets big:** lower `THREED_SPIN_FRAMES`,
  `THREED_SPIN_RES` or `THREED_SPIN_SIZE`. Cost is one full render per frame, and the GIF is
  roughly proportional to frames x pixels.
- **The turntable will not animate in a viewer:** it is a plain animated GIF, so it needs no
  JavaScript, but some static previewers only show the first frame. Open the file from
  `OUTPUT_DIR` in a browser to see it turn.
- **Why is 3D not drawn with Open3D?** Its offscreen renderer returns empty frames for point
  clouds beyond a few tens of thousands of points on this machine, and chunking does not help.
  Its in-notebook viewer is no better here: `draw_geometries` reported
  `WebRTCWindowSystem: HTTP handshake server disabled` and produced no output at all, which is
  why section 19 uses plotly for the in-notebook view and keeps Open3D for the desktop window
  (`SHOW_3D_VIEWER`) only. The z-buffer renderer in section 16 draws the same tiles at 10-21M
  points instead.
- **The interactive viewer in section 19 is blank:** an output that carries JavaScript only runs
  in a *trusted* notebook, so click *Trust Notebook* or run the cell yourself. It also needs
  `plotly` installed, which section 2 does.
- **The interactive view looks sparse:** it is a 60k-point sample of a 10M-point tile. Raise
  `PLOTLY_MAX_POINTS` (browsers slow down past a few hundred thousand), or set
  `PLOTLY_DETAIL_SIZE = None` to spread the same budget over the whole tile.
- **The notebook got much bigger:** the interactive cell inlines its own copy of plotly.js
  (~5 MB). Set `PLOTLY_INCLUDE_JS = "cdn"` to load it from the internet instead, or
  `SHOW_PLOTLY_VIEWER = False` to drop the cell to a one-line message.
- **Colours look wrong in the 3D views?** The palette is shared with the 2D class maps on
  purpose, so a colour means the same class in every figure.
- **Reruns are not bit-exact.** `cudnn.benchmark` autotunes kernels, so the first few epochs can
  differ slightly between runs. Three verification runs produced the same final model and the
  same held-out metrics. Set `torch.backends.cudnn.benchmark = False` in section 3 if you need
  bit-exact repeats.
- **Validation caveat:** the per-epoch validation split is random over blocks sampled from the
  training tiles, and neighbouring blocks can overlap spatially, so it is optimistic. The
  held-out tiles in section 13 are the metric to quote.
- **The new tile in section 21 comes out speckled, with Building sprayed across the road:** that
  file is ~50x denser than the training tiles, so `NEW_TILE_MAX_POINTS` defaults to 1.5M to
  match the density the blocks were built at. Section 21 has the measurements.
- **The new tile will not load:** `load_raw_scan` needs `X`/`Y`/`Z` columns and will use an
  `intensity` column if there is one, otherwise greyscale zeros.
- **The mesh in section 22 is a blobby green cauliflower:** that is Poisson filling in the
  space between leaf points, which is honest but ugly. Lower `MESH_DIST_MULT` to cut more of
  it away, or raise `MESH_TRIM`. Going too far turns the surface into confetti, and section 22
  prints the class mix at a few settings so the trade-off is visible.
- **The mesh looks much coarser than the point cloud on `L002`:** that tile is ~448 m across
  and the octree voxel is the diagonal over `2**depth`, so depth matters far more there than on
  the ~100 m Paris scan -- measured on `L002`, depth 10 gave 396k vertices and a gappy surface
  against 1.08M at depth 11, for the same 80 s. `MESH_DEPTH = "auto"` already picks depth 11
  there; force a smaller `MESH_VOXEL` to push it further, at roughly 8x the memory per level.
  Raising `MESH_MAX_POINTS` sharpens it a little more but is the expensive way to do it.
- **Meshing is slow:** normal orientation is the cost, about 80 s per 1.5M points, against
  seconds for Poisson itself. Lower `MESH_MAX_POINTS` or `MESH_ORIENT_K`, or drop tiles from
  `MESH_TILES`.
- **A mesh render looks flat:** the camera fits the whole tile into the frame, and a 500 m tile
  has little height next to its width. The bottom-left panel is the detail window
  (`MESH_DETAIL_SIZE`), which is where the geometry reads.
- **A mesh "sees" less than the point cloud:** correct, and deliberate. A mesh is a closed
  surface, so a canopy hides whatever is behind it and unscanned space stays a hole; the
  bottom-right panel of each figure is the same cloud from the same camera.

## 21. Last cell: a tile the model has never seen

`Paris_Luxembourg_6.ply` is not a Paris-Lille-3D tile -- it is a raw mobile-scan export
(`X, Y, Z, Intensity, Scan_Angle, GPS_Time, X_Sensor_Position, ...`) with no `reflectance`, no
RGB and no labels. That changes how it has to be fed in:

* `load_tile` takes colour from `reflectance` (Paris-Lille) or `red` / `green` / `blue`
  (Toronto). This file has neither, so loaded as-is it would hand the model three channels of
  black. `load_raw_scan` below sends the scanner's own `Intensity` through the same single
  greyscale channel the Paris-Lille tiles use, which is the closest match to what the model
  trained on;
* there is no ground truth in the file, so there is no accuracy to quote here. This is a look
  at what the model does on new ground rather than a measurement, and it is not zero-shot
  cross-city either: Paris tiles were part of training, so this is a new *area* of a city the
  model already knows;
* at ~2,000 points per square metre this export is far denser than the tiles the model trained
  on, and that matters. `predict_tile` feeds the model 1024-point neighbourhoods, so at full
  density each block covers about half a square metre -- roughly fifty times narrower than in
  training -- and the model starts calling road surface *Building*. Measured on this tile:

  | cloud | direct coverage | confidence | Road | Building | Natural |
  |---|---|---|---|---|---|
  | all 10M points | 19% | 0.727 | 35.9% | 39.4% | 16.4% |
  | thinned to 1.5M (what training tiles were capped to) | 65% | 0.807 | 53.5% | 21.9% | 16.1% |

  The thinned run is also visibly the cleaner render: red no longer sprays across the road.
  `NEW_TILE_MAX_POINTS` therefore defaults to `1_500_000`; set it to `None` to predict every
  point instead. Tightening `NEW_TILE_SPACING` is not a fix -- at 0.5 m and 0.25 m the direct
  coverage rises to 47% and 82%, but the class mix barely moves (Building stays at 39%), because
  the block geometry is what changed.

The cell writes `pred_Paris_Luxembourg_6.npy` and `threed_interactive_Paris_Luxembourg_6.html`
into `OUTPUT_DIR` alongside the other results.

In [ ]:
# --------------------------------------------------------------------------- #
# A tile that never took part in training, never mind the notebook.
# --------------------------------------------------------------------------- #
NEW_TILE_PATH = DATA_ROOT / "Paris_Luxembourg_6.ply"
NEW_TILE_SPACING = 1.0        # metres between block anchors, as used for the Paris-Lille tiles
NEW_TILE_MAX_POINTS = 1_500_000   # match the training density; None = use every point
NEW_TILE_RES = 1100           # pixels for the top-down map
NEW_TILE_VIEWER = True        # also embed the rotatable scene for this tile


def load_raw_scan(path, max_points=None, seed=SEED, verbose=True):
    '''Raw scanner export (X, Y, Z, Intensity, ...) -> the features load_tile would build.

    load_tile keys colour off `reflectance` / `red`, and this export has neither, so on its
    own it would feed the model three all-zero colour channels. Here the scanner's own
    `Intensity` goes through the same single greyscale channel the Paris-Lille tiles use.
    '''
    ply = PlyData.read(str(path))
    v = ply["vertex"].data
    low = {f.lower(): f for f in v.dtype.names}
    for need in ("x", "y", "z"):
        if need not in low:
            raise ValueError(f"missing {need.upper()} in {path}: {list(v.dtype.names)}")
    n = len(v)
    if max_points is not None and n > max_points:
        keep = np.sort(np.random.default_rng(seed).choice(n, max_points, replace=False))
    else:
        keep = np.arange(n)
    world = np.column_stack([v[low["x"]][keep], v[low["y"]][keep],
                             v[low["z"]][keep]]).astype(np.float64)
    if "intensity" in low:
        ch = np.asarray(v[low["intensity"]][keep], np.float32)
        if ch.max(initial=0) > 1:
            ch /= 255.0
    else:
        ch = np.zeros(len(keep), np.float32)
        print("  no intensity field: feeding greyscale zeros")
    if verbose:
        print(f"  {path.name}: {n:,} points, fields {list(v.dtype.names)}"
              + (f", thinning to {len(keep):,}" if len(keep) < n else ""))
    rgb = np.repeat(np.clip(ch, 0, 1)[:, None], 3, axis=1)
    feats = np.concatenate([world - world.mean(axis=0), rgb], axis=1).astype(np.float32)
    del ply, v
    return world, feats


if not NEW_TILE_PATH.exists():
    print(f"{NEW_TILE_PATH} not found -- put the file next to the notebook and rerun this cell")
else:
    print(f"=== {NEW_TILE_PATH.name} ===")
    t0 = time.time()
    new_world, new_feats = load_raw_scan(NEW_TILE_PATH, max_points=NEW_TILE_MAX_POINTS)
    density = len(new_feats) / max(np.ptp(new_world[:, 0]) * np.ptp(new_world[:, 1]), 1e-9)
    print(f"  {len(new_feats):,} points | X {new_world[:, 0].min():.0f}..{new_world[:, 0].max():.0f}"
          f" | Y {new_world[:, 1].min():.0f}..{new_world[:, 1].max():.0f}"
          f" | Z {new_world[:, 2].min():.1f}..{new_world[:, 2].max():.1f}"
          f" | {density:.0f} pts/m2 | loaded in {time.time() - t0:.0f}s")

    t0 = time.time()
    new_pred, new_prob, new_cov = predict_tile(new_feats, model, spacing=NEW_TILE_SPACING,
                                               verbose=False)
    dt = time.time() - t0
    share = np.bincount(new_pred, minlength=NUM_CLASSES) / max(len(new_pred), 1)
    print(f"  predicted in {dt:.0f}s ({len(new_pred) / dt / 1e3:.0f}k pts/s) | direct coverage "
          f"{new_cov.mean() * 100:.1f}% | mean confidence {new_prob.max(axis=1).mean():.3f}")
    for i in np.argsort(-share):
        if share[i] >= 0.002:
            print(f"    {CLASS_NAMES[i]:12} {share[i] * 100:5.1f}%")
    np.save(OUTPUT_DIR / f"pred_{NEW_TILE_PATH.stem}.npy", new_pred)

    # ---- what it looks like -------------------------------------------------
    fig, ax = plt.subplots(1, 2, figsize=(15.5, 8.6),
                           gridspec_kw={"width_ratios": [1.0, 1.22]})
    ax[0].imshow(render_class_map(new_world[:, 0], new_world[:, 1], new_pred,
                                  resolution=NEW_TILE_RES), interpolation="nearest")
    ax[0].set_title(f"{NEW_TILE_PATH.stem} predicted, top down ({len(new_pred):,} points)",
                    fontsize=12)
    box = detail_crop(new_world, new_pred, size=90.0)
    yaw = principal_yaw(new_world, box)
    ax[1].imshow(render_3d(new_world, new_pred, yaw=yaw, pitch=30, resolution=(1000, 840)))
    ax[1].set_title(f"the same tile in 3D (yaw {yaw:.0f}, pitch 30), every point drawn",
                    fontsize=12)
    for a in ax:
        a.set_xticks([])
        a.set_yticks([])
    fig.legend(handles=[Patch(facecolor=PALETTE[i], label=f"{i}: {CLASS_NAMES[i]}")
                        for i in range(NUM_CLASSES)],
               loc="lower center", ncol=NUM_CLASSES, frameon=False, fontsize=11)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(15, 6.2))
    for a, pitch, tag in zip(axes, (12, 55), ("street level", "from above")):
        a.imshow(render_3d(new_world, new_pred, yaw=yaw, pitch=pitch, resolution=(1000, 760)))
        a.set_title(f"{NEW_TILE_PATH.stem} predicted {tag} (pitch {pitch})", fontsize=12)
        a.set_xticks([])
        a.set_yticks([])
    plt.tight_layout()
    plt.show()

    if NEW_TILE_VIEWER and PLOTLY_AVAILABLE:
        from IPython.display import HTML, display
        title = (f"{NEW_TILE_PATH.stem} predicted in 3D -- drag to orbit, wheel to zoom "
                 f"({min(PLOTLY_MAX_POINTS or len(new_pred), len(new_pred)):,} of "
                 f"{len(new_pred):,} points)")
        html, counts = plotly_3d_html(new_world, new_pred, title)
        out = OUTPUT_DIR / f"threed_interactive_{NEW_TILE_PATH.stem}.html"
        out.write_text(html, encoding="utf-8")
        print(f"  {int(counts.sum()):,} points in {int((counts > 0).sum())} class traces -> "
              f"{out.name} ({len(html) / 1e6:.2f} MB)")
        display(HTML(html))

## 22. The segmentation as a surface

Every section so far draws labels as **points**. This one turns the same labels into
**geometry**: the cloud is reconstructed into a triangle mesh, and each mesh vertex inherits
the class of the nearest labelled point. The result is a coloured `PLY` that opens in
MeshLab, CloudCompare or Blender with the segmentation stored per vertex, and the renders
below are that mesh -- lit, from the same kind of camera as section 17.

`mesh_from_labels` samples the tile to `MESH_MAX_POINTS`, estimates normals at a radius of
`MESH_NORMAL_SCALE` x the tile's median point spacing (so it adapts to density instead of
carrying a hard-coded radius), orients them consistently, runs Poisson reconstruction, and
transfers classes by nearest neighbour in a KD-tree -- the same kind of transfer the model's
own features use. The octree depth matters more than anything else here and no single value
suits both a ~100 m Paris tile and a ~450 m Toronto one, because the voxel size is the tile
diagonal over `2**depth`. `MESH_DEPTH` therefore defaults to `"auto"`: it picks the depth whose
voxel is nearest `MESH_VOXEL` (10 cm) for the tile in hand -- depth 10 for the Paris scan,
depth 11 for `L002` -- and the chosen depth and voxel are printed for each tile. On `L002`,
which is 448 m across, depth 10 gives 396k vertices and a surface with visible gaps while
depth 11 gives 1.08M and reads as a surface, and both take about 80 s for 1.5M points: the
octree level is not what makes this slow.

Two cuts keep the surface honest:

- **`MESH_TRIM`** drops the lowest-density fraction of Poisson's own vertices. The octree's
  density is its confidence, and the tail is the thin fringe that hangs off the data.
- **`MESH_DIST_MULT`** drops any vertex further than that many median point spacings from a
  real scan point. Poisson *fills in* what the scanner never saw, which is exactly what makes
  a surface out of a cloud -- and also what inflates a tree canopy into a closed blob. This is
  the cut that removes the ballooning. Loosen it and the blobs return; tighten it and the mesh
  turns into confetti. The printout gives the class mix at a few settings so the choice is
  visible rather than asserted.

**Reading the renders.** A mesh is an opaque surface, so unlike every point view it cannot be
seen through: where a canopy stands between the camera and a facade, the canopy wins, and
where the scanner never looked (behind walls, up under eaves) there is simply a hole. The
bottom-right panel is the cloud the mesh came from, same camera and same labels, so that
difference is visible instead of claimed.

**Why this needs its own renderer.** `render_3d` splats points. `render_mesh_3d` samples each
triangle in screen space *in proportion to its projected area* -- a 40 m ground triangle gets
hundreds of samples, a sliver gets one, `MESH_SAMPLES_PER_PIXEL` samples per pixel of surface --
then z-buffers and majority-votes exactly as `render_3d` does. It also shades the result with a
Lambert pass driven by the mesh normals, and that shading is what makes a facade read as a wall
instead of a red puddle. That sample count is not a cosmetic dial: at one sample per triangle the
golden-ratio offsets repeat from triangle to triangle, the surface moires, and the render comes
out speckled with black -- the values here are the ones that fill it in. Note that a mesh vertex
distribution is area-weighted, not point-weighted, so the mesh mix below legitimately differs
from the cloud mix: foliage has a lot of surface per scan point.

Cost is dominated by normal orientation: roughly 80 s per 1.5M-point tile, against seconds for
Poisson itself. `MESH_TILES`, `MESH_MAX_POINTS` and `MESH_ORIENT_K` are the knobs.

In [ ]:
# --------------------------------------------------------------------------- #
# 22. The segmentation as a surface: reconstruction that keeps the classes.
# --------------------------------------------------------------------------- #
# open3d is imported here, not in section 3, because this is the only place that needs it and
# its renderer is unusable on this machine anyway -- what is used is its CPU geometry.
import open3d as o3d


def mesh_from_labels(xyz, labels, depth=MESH_DEPTH, max_points=MESH_MAX_POINTS,
                     trim=MESH_TRIM, nn_scale=MESH_NORMAL_SCALE, orient_k=MESH_ORIENT_K,
                     dist_mult=MESH_DIST_MULT, seed=SEED, verbose=True):
    '''Labelled points -> (mesh in world coordinates, per-vertex class, stats).

    Poisson reconstruction fills in everything the scanner missed, which is what makes a
    surface out of a cloud -- and also what turns a tree canopy into a blob. Poisson's own
    low-density vertices (`trim`) and vertices too far from any real point (`dist_mult`) are
    both dropped, then each surviving vertex takes the class of the nearest labelled point.
    '''
    xyz = np.asarray(xyz, np.float64)
    labels = np.asarray(labels, np.int64)
    n = len(xyz)
    keep = (np.sort(np.random.default_rng(seed).choice(n, max_points, replace=False))
            if (max_points is not None and n > max_points) else np.arange(n))
    P = np.ascontiguousarray(xyz[keep])
    L = labels[keep]
    centre = P.mean(axis=0)
    P -= centre        # world offsets run to 10^6 m; Poisson is happier near the origin

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(P)
    t0 = time.time()
    med = float(np.median(cKDTree(P).query(P, k=2, workers=-1)[0][:, 1]))
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=nn_scale * med, max_nn=30))
    pcd.orient_normals_consistent_tangent_plane(orient_k)     # the expensive step
    t_prep = time.time() - t0

    # One fixed depth cannot suit both a 100 m Paris tile and a 450 m Toronto one: the voxel
    # size is the diagonal over 2^depth, so "auto" picks the depth that lands near MESH_VOXEL.
    extent = float(np.linalg.norm(P.max(axis=0) - P.min(axis=0)))
    dep = (int(np.clip(round(math.log2(max(extent, 1.0) / MESH_VOXEL)),
                         MESH_DEPTH_MIN, MESH_DEPTH_MAX))
           if isinstance(depth, str) else int(depth))
    voxel = extent / 2.0 ** dep

    t0 = time.time()
    mesh, density = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=dep)
    t_poisson = time.time() - t0
    density = np.asarray(density)
    n_raw = len(np.asarray(mesh.vertices))

    def drop(mask, why):
        mesh.remove_vertices_by_mask(np.asarray(mask))
        mesh.remove_degenerate_triangles()
        mesh.remove_unreferenced_vertices()
        if verbose:
            print(f"    dropped {int(np.asarray(mask).sum()):,} vertices {why}")

    drop(density <= np.quantile(density, trim), f"in the bottom {trim * 100:.0f}% of Poisson density")
    far = cKDTree(P).query(np.asarray(mesh.vertices), k=1, workers=-1)[0] > dist_mult * med
    drop(far, f"further than {dist_mult} x {med * 100:.1f} cm from any scan point")
    mesh.compute_vertex_normals()

    vcls = L[cKDTree(P).query(np.asarray(mesh.vertices), k=1, workers=-1)[1]]
    mesh.vertex_colors = o3d.utility.Vector3dVector(PALETTE[vcls])
    mesh.translate(centre)                     # back to world coordinates, ready to export

    stats = dict(points=len(P), spacing=med, raw=n_raw, vertices=len(vcls),
                 triangles=len(np.asarray(mesh.triangles)), prep=t_prep, poisson=t_poisson,
                 depth=dep, voxel=voxel)
    if verbose:
        print(f"    {stats['points']:,} pts -> {n_raw:,} raw vertices -> {stats['vertices']:,} "
              f"vertices, {stats['triangles']:,} triangles | median spacing {med * 100:.1f} cm | "
              f"depth {dep} (voxel {voxel * 100:.0f} cm) | normals+orientation {t_prep:.0f}s, "
              f"poisson {t_poisson:.0f}s")
    return mesh, vcls, stats


@torch.no_grad()
def render_mesh_3d(mesh, vcls, yaw=0.0, pitch=30.0, resolution=MESH_RES, fov=45.0,
                   distance=2.6, background=0.07, spp=MESH_SAMPLES_PER_PIXEL, kmax=256,
                   ambient=0.45, sun=(0.35, -0.45, 0.82), slab=1.02, crop=None):
    '''Rasterise a segmented mesh as a lit class map: the mesh counterpart of render_3d.

    Every triangle is sampled in screen space in proportion to its projected area, `spp`
    samples per pixel of surface, so a large ground triangle gets many samples and a sliver
    gets one, and the samples are then z-buffered and majority-voted exactly like points are
    in render_3d. Vertex normals drive a Lambert pass, which is what separates a wall from
    the road beneath it. `crop` is a window in world units, and like render_3d it defines the
    frame, not just a filter.
    '''
    W, H = int(resolution[0]), int(resolution[1])
    V = np.asarray(mesh.vertices, np.float64)
    T = np.asarray(mesh.triangles, np.int64)
    N = np.asarray(mesh.vertex_normals, np.float64)
    L = np.asarray(vcls, np.int64)
    inside = np.isfinite(V).all(axis=1)
    if crop is not None:
        (x0, y0), (x1, y1) = crop
        inside &= (V[:, 0] >= x0) & (V[:, 0] <= x1) & (V[:, 1] >= y0) & (V[:, 1] <= y1)
    T = T[inside[T].all(axis=1)]
    if len(T) == 0:
        return np.zeros((H, W, 3), np.float32)
    frame_v = V[inside] if crop is not None else V[np.unique(T)]
    centre = frame_v.mean(axis=0)
    scale = max(float(np.linalg.norm(frame_v - centre, axis=1).max()), 1e-9)

    eye_dir, right, up, forward = camera_basis(yaw, pitch)
    f0 = (H / 2.0) / math.tan(math.radians(fov) / 2.0)

    def project(pw):
        p = torch.as_tensor(((pw - centre) / scale).astype(np.float32), device=DEVICE)
        v = p - eye_dir * distance
        zc = v @ forward
        safe = zc.clamp_min(1e-4)          # only so that the divide cannot blow up
        return zc, W / 2.0 + f0 * (v @ right) / safe, H / 2.0 - f0 * (v @ up) / safe

    zc_f, px_f, py_f = project(frame_v)                 # fit the frame to the image
    front = (zc_f > 1e-4).cpu().numpy()
    pxn, pyn = px_f.cpu().numpy()[front], py_f.cpu().numpy()[front]
    if len(pxn) == 0:
        return np.zeros((H, W, 3), np.float32)
    fit = min(0.9 * W / max(float(np.ptp(pxn)), 1e-6),
              0.9 * H / max(float(np.ptp(pyn)), 1e-6))

    def fit_px(px, py):
        return W / 2.0 + (px - W / 2.0) * fit, H / 2.0 + (py - H / 2.0) * fit

    zc_t, px_t, py_t = project(V[T])
    fx, fy = fit_px(px_t.cpu().numpy(), py_t.cpu().numpy())
    area = 0.5 * np.abs((fx[:, 1] - fx[:, 0]) * (fy[:, 2] - fy[:, 0])
                        - (fx[:, 2] - fx[:, 0]) * (fy[:, 1] - fy[:, 0]))
    # `spp` samples per pixel of projected area. At one sample per triangle the golden-ratio
    # offsets repeat across neighbouring triangles and the surface moires, which reads as
    # black speckle; a few samples per pixel is what actually fills it in.
    k = np.clip(np.ceil(area * spp), 1, kmax).astype(np.int64)
    k[~np.isfinite(area)] = 1
    off = np.concatenate([[0], np.cumsum(k)])
    tri = np.repeat(np.arange(len(T)), k)
    j = np.arange(int(off[-1])) - off[tri]

    i = np.arange(kmax)                                 # golden-ratio points in a triangle
    su = np.sqrt((i * 0.6180339887498949 + 0.5) % 1.0)
    sv = (i * 0.7548776662466927 + 0.5) % 1.0
    bary = np.stack([1.0 - su, su * (1.0 - sv), su * sv], axis=1)[j]
    t = T[tri]
    b = bary[:, :, None]
    pos = V[t[:, 0]] * b[:, 0] + V[t[:, 1]] * b[:, 1] + V[t[:, 2]] * b[:, 2]
    nor = N[t[:, 0]] * b[:, 0] + N[t[:, 1]] * b[:, 1] + N[t[:, 2]] * b[:, 2]
    lab = L[t[np.arange(len(tri)), np.argmax(bary, axis=1)]]

    zs, pxs, pys = project(pos)
    fxs, fys = fit_px(pxs.cpu().numpy(), pys.cpu().numpy())
    zs = zs.cpu().numpy()
    valid = (zs > 1e-3) & (fxs >= 0) & (fxs < W) & (fys >= 0) & (fys < H)
    if not valid.any():
        return np.zeros((H, W, 3), np.float32)
    pix = torch.as_tensor(fys[valid].astype(np.int64) * W + fxs[valid].astype(np.int64),
                          device=DEVICE)
    zst = torch.as_tensor(zs[valid], device=DEVICE)
    depth = torch.full((H * W,), float("inf"), device=DEVICE)
    depth.scatter_reduce_(0, pix, zst, reduce="amin", include_self=True)
    near = zst <= depth[pix] * slab + 1e-6
    sel_pix = pix[near]
    sel_lab = torch.as_tensor(lab[valid], device=DEVICE)[near]

    votes = torch.bincount(sel_pix * NUM_CLASSES + sel_lab,
                           minlength=H * W * NUM_CLASSES).reshape(H * W, NUM_CLASSES)
    has = votes.sum(dim=1) > 0
    winner = votes.argmax(dim=1)

    nn = torch.as_tensor(nor[valid], device=DEVICE, dtype=torch.float32)
    nn = nn / nn.norm(dim=1, keepdim=True).clamp_min(1e-9)
    sun_v = torch.tensor(sun, device=DEVICE, dtype=torch.float32)
    lam = (nn[near] @ (sun_v / sun_v.norm())).abs()      # abs: orientation is per-vertex only
    lit = torch.zeros(H * W, device=DEVICE)
    lit.index_add_(0, sel_pix, lam)
    seen = torch.zeros(H * W, device=DEVICE)
    seen.index_add_(0, sel_pix, torch.ones_like(lam))
    shade = ambient + (1.0 - ambient) * (lit / seen.clamp_min(1.0))

    pal = torch.as_tensor(np.asarray(PALETTE, np.float32), device=DEVICE)
    img = torch.zeros(H * W, 3, device=DEVICE)
    img[has] = pal[winner[has]] * shade[has].unsqueeze(1)
    img[~has] = torch.tensor(background, device=DEVICE, dtype=torch.float32)
    return img.reshape(H, W, 3).clamp(0, 1).cpu().numpy()


if PLOTLY_AVAILABLE:
    def plotly_mesh_html(mesh, title, max_triangles=MESH_VIEWER_TRIANGLES, height=None):
        '''A rotatable plotly Mesh3d of the segmented surface (opt-in, `MESH_VIEWER`).

        A browser cannot draw a million triangles, so the surface is decimated to
        `MESH_VIEWER_TRIANGLES` first; vertex colours carry the classes across the decimation.
        '''
        import plotly.graph_objects as go
        import plotly.io as pio

        m = o3d.geometry.TriangleMesh(mesh)
        if max_triangles and len(np.asarray(m.triangles)) > max_triangles:
            m = m.simplify_quadric_decimation(max_triangles)
        V = np.asarray(m.vertices)
        F = np.asarray(m.triangles)
        C = np.clip(np.round(np.asarray(m.vertex_colors) * 255), 0, 255).astype(np.uint8)
        fig = go.Figure(go.Mesh3d(
            x=V[:, 0], y=V[:, 1], z=V[:, 2], i=F[:, 0], j=F[:, 1], k=F[:, 2],
            vertexcolor=["rgb(%d,%d,%d)" % tuple(c) for c in C],
            flatshading=False, hoverinfo="skip", name="segmented mesh"))
        fig.update_layout(
            title=title, height=PLOTLY_HEIGHT if height is None else height,
            margin=dict(l=0, r=0, t=34, b=0), paper_bgcolor="white",
            scene=dict(aspectmode="data", bgcolor="white",
                       camera=dict(eye=dict(x=1.5, y=1.5, z=0.8)),
                       xaxis=dict(visible=False), yaxis=dict(visible=False),
                       zaxis=dict(visible=False)))
        return pio.to_html(fig, full_html=False,
                           include_plotlyjs=("cdn" if PLOTLY_INCLUDE_JS == "cdn" else True),
                           config=dict(scrollZoom=True, displaylogo=False, responsive=True))


# ---- which tiles, from whatever has already been predicted -----------------
mesh_sources = {}
for _n in MESH_TILES:
    if _n in heldout:
        mesh_sources[_n] = (heldout[_n]["xyz"], heldout[_n]["pred"])
if NEW_TILE_PATH.stem in MESH_TILES:
    if "new_pred" in globals():
        mesh_sources[NEW_TILE_PATH.stem] = (new_world, new_pred)
    else:
        print(f"{NEW_TILE_PATH.stem}: run section 21 first -- no prediction in memory yet")
print(f"meshing {list(mesh_sources) or 'nothing'}")

mesh_stats = {}
for name, (xyz, lab) in mesh_sources.items():
    print(f"\n=== {name} ===")
    mesh, vcls, stats = mesh_from_labels(xyz, lab)
    mesh_stats[name] = stats

    cloud_share = np.bincount(lab, minlength=NUM_CLASSES) / max(len(lab), 1)
    mesh_share = np.bincount(vcls, minlength=NUM_CLASSES) / max(len(vcls), 1)
    print("    class mix                         mesh    cloud")
    for c in np.argsort(-cloud_share):
        if cloud_share[c] > 0.005:
            print(f"      {c}: {CLASS_NAMES[c]:13} {mesh_share[c] * 100:8.1f}% "
                  f"{cloud_share[c] * 100:7.1f}%")

    # the same kind of window sections 17-19 look through
    gt = heldout[name]["gt"] if (name in heldout and heldout[name]["gt"] is not None) else lab
    box = detail_crop(xyz, gt, size=MESH_DETAIL_SIZE) if MESH_DETAIL_SIZE else None
    yaw = principal_yaw(xyz, box)

    fig, axes = plt.subplots(2, 2, figsize=(15.5, 10.6))
    axes[0][0].imshow(render_mesh_3d(mesh, vcls, yaw=yaw, pitch=30))
    axes[0][0].set_title(f"{name} as a mesh -- {stats['vertices']:,} vertices, "
                         f"{stats['triangles']:,} triangles (pitch 30)", fontsize=12)
    axes[0][1].imshow(render_mesh_3d(mesh, vcls, yaw=yaw, pitch=55))
    axes[0][1].set_title("the same surface from above (pitch 55)", fontsize=12)
    if box is not None:
        axes[1][0].imshow(render_mesh_3d(mesh, vcls, yaw=yaw, pitch=22, crop=box))
        axes[1][0].set_title(f"{MESH_DETAIL_SIZE:.0f} m window at street level (pitch 22)",
                             fontsize=12)
    else:
        axes[1][0].imshow(render_mesh_3d(mesh, vcls, yaw=yaw, pitch=12))
        axes[1][0].set_title("street level (pitch 12)", fontsize=12)
    axes[1][1].imshow(render_3d(xyz, lab, yaw=yaw, pitch=30))
    axes[1][1].set_title("the cloud it came from, same camera -- a mesh is opaque",
                         fontsize=12)
    for a in axes.ravel():
        a.set_xticks([])
        a.set_yticks([])
    fig.legend(handles=[Patch(facecolor=PALETTE[i], label=f"{i}: {CLASS_NAMES[i]}")
                        for i in range(NUM_CLASSES)],
               loc="lower center", ncol=NUM_CLASSES, frameon=False, fontsize=11)
    plt.tight_layout(rect=(0, 0.035, 1, 1))
    plt.show()

    if MESH_EXPORT:
        out = OUTPUT_DIR / f"mesh_{name}.ply"
        o3d.io.write_triangle_mesh(str(out), mesh)
        print(f"    wrote {out.name} ({out.stat().st_size / 1e6:.1f} MB, classes as vertex colours)")

    if MESH_VIEWER and PLOTLY_AVAILABLE:
        from IPython.display import HTML, display
        html = plotly_mesh_html(mesh, f"{name} segmentation as a surface -- drag to orbit")
        out = OUTPUT_DIR / f"mesh_interactive_{name}.html"
        out.write_text(html, encoding="utf-8")
        print(f"    rotatable mesh -> {out.name} ({len(html) / 1e6:.1f} MB)")
        display(HTML(html))

if mesh_stats:
    tot = sum(s["vertices"] for s in mesh_stats.values())
    print(f"\n{len(mesh_stats)} mesh(es), {tot:,} vertices total -> {OUTPUT_DIR}")

## 23. Evaluation metrics, ablations, cost, and the weights themselves

Sections 11-13 printed two numbers per tile, OA and mIoU. This is the report:

- **full metric tables** -- per-class IoU, precision, recall and F1, plus mean accuracy,
  frequency-weighted IoU and Cohen's kappa, and a row-normalised confusion matrix;
- **ablations** -- retrained variants of the same recipe under paired seeds, an inference-time
  channel-occlusion test that needs no retraining, and the anchor-grid trade-off;
- **cost** -- parameter count, checkpoint size, training wall-clock, inference throughput and
  peak GPU memory;
- **the parameters themselves** -- every tensor with its shape and statistics, and the question
  of *bias*, which here is settled by the architecture rather than by convention.

On bias. Every convolution in the trunk is constructed with `bias=False`, because a batch-norm
layer always follows it and a bias sitting in front of batch-norm is algebraically redundant:
batch-norm subtracts the batch mean, so the bias is cancelled on every forward pass. The biases
that do exist are the batch-norm shifts and the 8 numbers in the final classification layer.
The counts below are measured off the live model, not asserted.

In [ ]:
# --------------------------------------------------------------------------- #
# 23a. Every standard metric, from one confusion matrix.
# --------------------------------------------------------------------------- #
def class_report(gt, pred, names=CLASS_NAMES):
    '''OA, mIoU, mAcc, frequency-weighted IoU, Cohen's kappa and the per-class scores.

    mIoU averages only over the classes present in the ground truth. A class with no points has
    an empty union; scoring it as 0 would drag the mean down through no fault of the model, so
    it is excluded instead (and its row in the report is skipped).
    '''
    K = len(names)
    gt = np.asarray(gt).reshape(-1)
    pred = np.asarray(pred).reshape(-1)
    ok = (gt >= 0) & (gt < K) & (pred >= 0) & (pred < K)
    gt, pred = gt[ok], pred[ok]
    conf = np.bincount(gt * K + pred, minlength=K * K).reshape(K, K).astype(np.float64)
    n = conf.sum()
    tp = np.diag(conf)
    fp = conf.sum(0) - tp
    fn = conf.sum(1) - tp
    support = conf.sum(1)
    union = tp + fp + fn
    iou = np.divide(tp, union, out=np.full(K, np.nan), where=union > 0)
    prec = np.divide(tp, tp + fp, out=np.full(K, np.nan), where=(tp + fp) > 0)
    rec = np.divide(tp, tp + fn, out=np.full(K, np.nan), where=(tp + fn) > 0)
    s = prec + rec
    f1 = np.divide(2 * prec * rec, s, out=np.full(K, np.nan), where=(s > 0) & np.isfinite(s))
    present = support > 0
    oa = float(tp.sum() / max(n, 1))
    pe = float((conf.sum(0) * conf.sum(1)).sum() / max(n * n, 1.0))
    return {"n": int(n), "support": support, "conf": conf, "iou": iou, "precision": prec,
            "recall": rec, "f1": f1, "present": present, "oa": oa,
            "miou": float(np.nanmean(iou[present])) if present.any() else float("nan"),
            "macc": float(np.nanmean(rec[present])) if present.any() else float("nan"),
            "fwiou": float(np.nansum(support * iou) / max(support.sum(), 1.0)),
            "kappa": float((oa - pe) / max(1.0 - pe, 1e-12))}


def print_report(rep, title, names=CLASS_NAMES):
    '''The per-class half of a report, as text, so every class is on the record.'''
    print(f"\n{title}  ({rep['n']:,} points)")
    print(f"  {'class':22}{'support':>12}{'IoU':>8}{'prec':>8}{'rec':>8}{'F1':>8}")
    for i in range(len(names)):
        if not rep["present"][i]:
            continue
        print(f"  {i}: {names[i]:19}{int(rep['support'][i]):>12,}"
              f"{rep['iou'][i]:>8.3f}{rep['precision'][i]:>8.3f}"
              f"{rep['recall'][i]:>8.3f}{rep['f1'][i]:>8.3f}")
    print(f"  {'macro over present':22}{'':>12}{rep['miou']:>8.3f}{'':>8}{rep['macc']:>8.3f}")
    print(f"  OA {rep['oa'] * 100:.2f}%  |  mIoU {rep['miou']:.4f}  |  mAcc {rep['macc']:.4f}  "
          f"|  frequency-weighted IoU {rep['fwiou']:.4f}  |  Cohen kappa {rep['kappa']:.4f}")


def confusion_figure(rep, title, names=CLASS_NAMES, ax=None):
    '''Row-normalised confusion matrix: where each true class actually went.'''
    keep = rep["present"]
    m = rep["conf"][np.ix_(keep, keep)].astype(np.float64)
    m = m / np.maximum(m.sum(1, keepdims=True), 1)
    lbl = [names[i] for i in range(len(names)) if keep[i]]
    if ax is None:
        fig, ax = plt.subplots(figsize=(1.6 + 0.8 * len(lbl), 0.8 * len(lbl) + 2.6))
    im = ax.imshow(m, cmap="viridis", vmin=0.0, vmax=1.0)
    ax.set_xticks(range(len(lbl)))
    ax.set_xticklabels(lbl, rotation=45, ha="right", fontsize=10)
    ax.set_yticks(range(len(lbl)))
    ax.set_yticklabels(lbl, fontsize=10)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    for i in range(len(lbl)):
        for j in range(len(lbl)):
            if m[i, j] >= 0.005:
                ax.text(j, i, f"{m[i, j] * 100:.0f}", ha="center", va="center", fontsize=9,
                        color="white" if m[i, j] < 0.55 else "black")
    ax.set_title(title, fontsize=12)
    ax.figure.colorbar(im, ax=ax, fraction=0.046, label="fraction of the true class")
    return ax


def per_class_bars(rep, title, names=CLASS_NAMES, ax=None):
    keep = rep["present"]
    lbl = [names[i] for i in range(len(names)) if keep[i]]
    x = np.arange(len(lbl))
    if ax is None:
        fig, ax = plt.subplots(figsize=(1.6 * len(lbl) + 3.6, 4.8))
    w = 0.2
    for k, (key, col) in enumerate(zip(("iou", "precision", "recall", "f1"),
                                       ("#4477aa", "#ee6677", "#228833", "#ccbb44"))):
        ax.bar(x + (k - 1.5) * w, rep[key][keep], w, label=key, color=col,
               edgecolor="white", linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(lbl, rotation=30, ha="right", fontsize=10)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("score")
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_title(title, fontsize=12)
    ax.legend(ncol=4, fontsize=10)
    return ax


print("metric helpers ready")

In [ ]:
# --------------------------------------------------------------------------- #
# 23b. The report itself: validation blocks, then each held-out tile.
# --------------------------------------------------------------------------- #
from IPython.display import display


@torch.no_grad()
def blocks_predict(x):
    out = []
    for s in range(0, len(x), INFER_BLOCK_BATCH):
        out.append(model(x[s:s + INFER_BLOCK_BATCH]).argmax(-1).reshape(-1).cpu().numpy())
    return np.concatenate(out)


reports = {"validation blocks": class_report(VA_LABELS.reshape(-1).cpu().numpy(),
                                            blocks_predict(VA_POINTS))}
for _name, _h in heldout.items():
    if _h["gt"] is not None:
        reports[f"{_name} (held out)"] = class_report(_h["gt"].astype(np.int64), _h["pred"])

print("=== headline metrics ===")
display(pd.DataFrame([{"tile": k, "points": f"{r['n']:,}", "OA %": f"{r['oa'] * 100:.2f}",
                       "mIoU": f"{r['miou']:.4f}", "mAcc": f"{r['macc']:.4f}",
                       "freq-w IoU": f"{r['fwiou']:.4f}", "kappa": f"{r['kappa']:.4f}",
                       "classes present": int(r["present"].sum())}
                      for k, r in reports.items()]))

for _name, _rep in reports.items():
    print_report(_rep, _name)

held_keys = [k for k in reports if k.endswith("(held out)")]
for _name in held_keys:
    confusion_figure(reports[_name], f"{_name}: confusion matrix, % of each true class")
    plt.tight_layout(); plt.show()

if held_keys:
    fig, axes = plt.subplots(len(held_keys), 1, figsize=(15, 6.6 * len(held_keys)))
    for _a, _name in zip(np.atleast_1d(axes), held_keys):
        per_class_bars(reports[_name], _name, ax=_a)
    plt.tight_layout(); plt.show()

In [ ]:
# --------------------------------------------------------------------------- #
# 23c. What it costs.
# --------------------------------------------------------------------------- #
def human_bytes(b):
    for unit in ("B", "KiB", "MiB", "GiB"):
        if b < 1024 or unit == "GiB":
            return (f"{b:.0f} B" if unit == "B" else f"{b:.1f} {unit}")
        b /= 1024


_param = list(model.named_parameters())
param_total = sum(p.numel() for _, p in _param)
param_1d = sum(p.numel() for _, p in _param if p.ndim == 1)
steps_now = max(1, len(TR_POINTS) // BATCH_SIZE)
ms_step = train_minutes * 60 * 1000 / max(len(history["loss"]) * steps_now, 1)
best_ep = int(np.argmax(history["val_miou"])) + 1

print("=== the model ===")
print(f"  parameters                              {param_total:>10,}  (fp32: "
      f"{human_bytes(param_total * 4)})")
print(f"  ... convolution / linear weights        {param_total - param_1d:>10,}  "
      f"({(param_total - param_1d) / param_total * 100:.2f}%)")
print(f"  ... 1-D parameters (norm scales/shifts, head bias)"
      f"  {param_1d:>4,}  ({param_1d / param_total * 100:.2f}%)")
print(f"  buffers (batch-norm running stats)      "
      f"{sum(b.numel() for b in model.buffers()):>10,}")
print(f"  checkpoint on disk                      "
      f"{human_bytes(MODEL_PATH.stat().st_size):>10}  -> {MODEL_PATH.name}")
print(f"  input   {NUM_POINTS} points x 6 channels (tile-centred XYZ + RGB), output "
      f"{NUM_CLASSES} logits per point")

print("\n=== training ===")
print(f"  epochs run {len(history['loss'])} of at most {EPOCHS} (early stopping, "
      f"patience {PATIENCE}); best at epoch {best_ep}")
print(f"  {len(TRAIN_TILES)} training tiles -> {sum(t['blocks'] for t in tile_report):,} blocks, "
      f"split {len(TR_POINTS):,} train / {len(VA_POINTS):,} validation")
print(f"  {steps_now} steps of batch {BATCH_SIZE} per epoch, {ms_step:.1f} ms/step "
      f"({ms_step / BATCH_SIZE:.2f} ms/block)")
print(f"  wall clock {train_minutes:.2f} min, of which "
      f"{len(history['loss']) - best_ep} epochs came after the best checkpoint")
print(f"  best validation mIoU {max(history['val_miou']):.4f} "
      f"(OA {history['val_oa'][best_ep - 1] * 100:.2f}%)")

print("\n=== inference ===")
display(pd.DataFrame([{"tile": k, "points": f"{h['n']:,}", "anchor grid (m)": h["spacing"],
                       "seconds": f"{h.get('seconds', float('nan')):.1f}",
                       "k points/s": f"{h['n'] / max(h.get('seconds', 1), 1e-9) / 1e3:.0f}",
                       "direct coverage": f"{h['coverage'].mean() * 100:.1f}%",
                       "mean confidence": f"{h['conf'].mean():.3f}"}
                      for k, h in heldout.items()]))
print("  direct coverage is the share of points that were actually inside a predicted block; the\n"
      "  rest inherit the nearest covered point's label verbatim. A whole tile is predicted in\n"
      "  spatial chunks of INFER_TILE_POINTS and a chunk only sees its own points, so the two big\n"
      "  held-out tiles land near a third -- their metrics lean on that fill, and neither number\n"
      "  is a fair estimate of what the model would do with the whole tile in one pass.")
if torch.cuda.is_available():
    print(f"  peak GPU memory this session "
          f"{torch.cuda.max_memory_allocated() / 2 ** 30:.2f} GiB of "
          f"{torch.cuda.get_device_properties(0).total_memory / 2 ** 30:.1f} GiB "
          f"({torch.cuda.get_device_name(0)})")

ax = plt.subplots(figsize=(9.6, 4.6))[1]
ax.plot(history["loss"], label="train loss")
ax.plot(history["val_loss"], label="validation loss")
ax.axvline(best_ep - 1, color="0.6", ls=":", lw=1.2)
ax.set_xlabel("epoch")
ax.set_ylabel("weighted cross-entropy")
ax.grid(True, alpha=0.3)
ax2 = ax.twinx()
ax2.plot([v * 100 for v in history["val_miou"]], color="#228833", ls="--",
         label="val mIoU x 100")
ax2.set_ylabel("validation mIoU x 100", color="#228833")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="center right", fontsize=10)
ax.set_title(f"training, best checkpoint at epoch {best_ep} "
             f"(mIoU {max(history['val_miou']):.3f})", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# --------------------------------------------------------------------------- #
# 23d. Ablation 1: retrain the same recipe with one thing switched off.
# --------------------------------------------------------------------------- #
# Retraining is the only honest way to attribute a score to a component, and it is also the
# expensive one. Each variant therefore runs ABL_EPOCHS epochs rather than the full EPOCHS, and
# the runs are paired: same seed, same initial weights, same batch order every epoch (from a
# dedicated CUDA generator), so the only difference between two rows is the switch. Read the
# table as *how fast each configuration learns*; it is not converged quality. The full row is
# byte-for-byte the recipe section 10 ran, just stopped early.
RUN_ABLATIONS = True     # sections 23d and 23f cost a few minutes; False skips both
ABL_EPOCHS = 6


def train_variant(tag, augment=True, zero=None, weighted=True, smooth=LABEL_SMOOTHING,
                  epochs=ABL_EPOCHS, seed=SEED):
    '''One short run of section 10's training loop, with exactly one thing changed.'''
    torch.manual_seed(seed)
    m = PointNet2SSG(in_channels=6, num_classes=NUM_CLASSES).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=(CLASS_WEIGHTS if weighted else None),
                               label_smoothing=smooth)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    nb_ = len(TR_POINTS)
    spe = max(1, nb_ // BATCH_SIZE)
    warm, tot = spe * 3, spe * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (
        (s + 1) / max(1, warm) if s < warm else
        0.5 * (1 + math.cos(math.pi * min((s - warm) / max(1, tot - warm), 1.0)))))
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(seed + 7)
    curve, oas, t0 = [], [], time.time()
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(nb_, device=DEVICE, generator=gen)
        for s in range(0, nb_ - BATCH_SIZE + 1, BATCH_SIZE):
            sel = perm[s:s + BATCH_SIZE]
            pts, labs = TR_POINTS[sel], TR_LABELS[sel]
            if augment:
                pts, labs = augment_batch(pts, labs)
            if zero is not None:
                pts = pts.clone()
                pts[:, :, zero] = 0.0
            opt.zero_grad(set_to_none=True)
            loss = crit(m(pts).reshape(-1, NUM_CLASSES), labs.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()
        oa, ious, _ = evaluate_blocks(m, VA_POINTS, VA_LABELS)
        curve.append(float(np.nanmean(ious)))
        oas.append(float(oa))
        print(f"    {tag:32} epoch {ep + 1}/{epochs} | val mIoU {curve[-1]:.4f} "
              f"| val OA {oas[-1] * 100:5.2f}%", flush=True)
    k = int(np.argmax(curve))
    return {"variant": tag, "best mIoU": curve[k], "OA at best %": oas[k] * 100,
            "last-epoch mIoU": curve[-1], "seconds": time.time() - t0, "curve": curve}


abl_runs = []
if RUN_ABLATIONS:
    for _tag, _kw in [("full (as delivered)", {}),
                      ("no augmentation", dict(augment=False)),
                      ("no colour (RGB = 0)", dict(zero=slice(3, 6))),
                      ("no geometry (XYZ = 0)", dict(zero=slice(0, 3))),
                      ("uniform loss (no class weights)", dict(weighted=False)),
                      ("no label smoothing", dict(smooth=0.0))]:
        print(f"\n  training variant: {_tag}")
        abl_runs.append(train_variant(_tag, **_kw))
    _base = abl_runs[0]["best mIoU"]
    display(pd.DataFrame([{k: v for k, v in r.items() if k != "curve"}
                          | {"delta mIoU vs full": r["best mIoU"] - _base}
                          for r in abl_runs]).round(4))
    fig, ax = plt.subplots(figsize=(10, 5))
    for r in abl_runs:
        ax.plot(range(1, len(r["curve"]) + 1), r["curve"], marker="o", ms=4, label=r["variant"])
    ax.set_xlabel("epoch")
    ax.set_ylabel("validation mIoU")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9.5)
    ax.set_title(f"ablation: {ABL_EPOCHS} epochs, paired seeds and batch order", fontsize=12)
    plt.tight_layout(); plt.show()
    print("how to read the table: six epochs is short enough that a regulariser can still be a\n"
          "cost rather than a saving. Switching augmentation off is the clearest case -- it ends\n"
          "slightly *above* the full recipe, on the same data and the same schedule, because the\n"
          "run never gets far enough for the extra variety to pay for itself. The two input\n"
          "groups dominate by an order of magnitude: zeroing RGB costs about 0.40 mIoU and\n"
          "zeroing XYZ about 0.47, both close to what ignoring the input entirely would give, so\n"
          "neither channel is decorative. Class weighting is worth roughly 0.13 mIoU (section 7's\n"
          "imbalance argument, confirmed by retraining), and label smoothing lands inside the\n"
          "run-to-run noise of a six-epoch schedule -- it is a stability choice, not an accuracy\n"
          "one. None of these rows is converged; EPOCHS is the setting that would give that.")
else:
    print("RUN_ABLATIONS is False: the retrained variants were skipped")

In [ ]:
# --------------------------------------------------------------------------- #
# 23e. Ablation 2: break the input at inference only. No retraining.
# --------------------------------------------------------------------------- #
# The retrained rows above answer "does the model need this channel in order to learn it".
# These answer "does the trained model actually read it", which is a different question and
# costs one forward pass per row. Measured on the validation blocks, already resident on the
# GPU. The last row is a control: shuffling the points inside a block cannot change a
# permutation-invariant set network, so it must come back at exactly zero.
def _zero_colour(p):
    q = p.clone(); q[..., 3:6] = 0.0; return q


def _zero_geometry(p):
    q = p.clone(); q[..., 0:3] = 0.0; return q


def _zero_height(p):
    q = p.clone(); q[..., 2] = 0.0; return q


def _shuffle_colour(p):
    perm = torch.argsort(torch.rand(p.shape[0], p.shape[1], device=p.device), dim=1)
    q = p.clone()
    q[..., 3:6] = torch.gather(p[..., 3:6], 1, perm[..., None].expand(-1, -1, 3))
    return q


def _shuffle_points(p):
    perm = torch.argsort(torch.rand(p.shape[0], p.shape[1], device=p.device), dim=1)
    return torch.gather(p, 1, perm[..., None].expand(-1, -1, p.shape[-1]))


torch.manual_seed(SEED)
occl = []
for _tag, _fn in [("intact", lambda p: p),
                  ("colour zeroed", _zero_colour),
                  ("colour shuffled within the block", _shuffle_colour),
                  ("geometry zeroed", _zero_geometry),
                  ("height (z) zeroed", _zero_height),
                  ("points shuffled (control)", _shuffle_points)]:
    _oa, _ious, _ = evaluate_blocks(model, _fn(VA_POINTS), VA_LABELS)
    occl.append({"input": _tag, "val mIoU": float(np.nanmean(_ious)), "val OA %": _oa * 100})
_o0 = occl[0]["val mIoU"]
for _r in occl:
    _r["delta mIoU"] = _r["val mIoU"] - _o0
display(pd.DataFrame(occl).round(4))

fig, ax = plt.subplots(figsize=(9.6, 4.4))
_rv = occl[::-1]
ax.barh([r["input"] for r in _rv], [r["delta mIoU"] for r in _rv], color="#4477aa")
ax.axvline(0, color="0.3", lw=1)
ax.set_xlabel("change in validation mIoU when the input is broken")
ax.grid(True, axis="x", alpha=0.3)
ax.set_title("what the trained network actually reads", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# --------------------------------------------------------------------------- #
# 23f. Ablation 3: the inference anchor grid.
# --------------------------------------------------------------------------- #
# Block size is fixed by the model, but the grid the blocks are anchored on is free: a finer
# grid means more blocks, more overlap and more time. Measured on a 1M-point subsample of the
# held-out L002 so the sweep stays cheap.
#
# Note the levels. This subsample fits inside a single inference chunk, so almost every point
# here is directly voted on (82.6% at 2 m), whereas the full L002 tile is five chunks and only
# 31.4% of its points are (see section 23c). That is why the numbers below sit *above* section
# 13's L002 row rather than below it. Read the shape of the curve, not its absolute level.
SPACING_VALUES = (1.0, 2.0, 4.0, 8.0)

if RUN_ABLATIONS:
    _l002 = dict((n, p) for n, p, _s in TEST_TILES)["L002"]
    sp_world, sp_feats, sp_labs, _ = load_tile(_l002, max_points=1_000_000, verbose=False)
    sp_rows = []
    for sp in SPACING_VALUES:
        t0 = time.time()
        p, _, cov = predict_tile(sp_feats, model, spacing=sp, verbose=False)
        dt = time.time() - t0
        rep = class_report(sp_labs, p)
        sp_rows.append({"anchor grid (m)": sp, "OA %": rep["oa"] * 100, "mIoU": rep["miou"],
                        "mAcc": rep["macc"], "kappa": rep["kappa"], "seconds": dt,
                        "coverage %": cov.mean() * 100})
        print(f"  spacing {sp:4.1f} m | OA {rep['oa'] * 100:5.2f}% | mIoU {rep['miou']:.4f} | "
              f"mAcc {rep['macc']:.4f} | kappa {rep['kappa']:.4f} | {dt:5.1f}s | coverage "
              f"{cov.mean() * 100:.1f}%", flush=True)
    display(pd.DataFrame(sp_rows).round(4))
    _xs = [r["anchor grid (m)"] for r in sp_rows]
    fig, ax = plt.subplots(figsize=(9, 4.6))
    ax.plot(_xs, [r["OA %"] for r in sp_rows], marker="o", label="OA %")
    ax.plot(_xs, [r["mIoU"] * 100 for r in sp_rows], marker="s", label="mIoU x 100")
    ax.set_xscale("log", base=2)
    ax.set_xticks(_xs)
    ax.set_xticklabels([f"{v:g}" for v in _xs])
    ax.set_xlabel("anchor grid spacing (m, log scale)")
    ax.set_ylabel("percent")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="center left")
    ax2 = ax.twinx()
    ax2.plot(_xs, [r["seconds"] for r in sp_rows], color="#ccbb44", ls="--", marker="^")
    ax2.set_ylabel("seconds", color="#ccbb44")
    ax.set_title("L002 (1M-point subsample): accuracy against anchor spacing", fontsize=12)
    plt.tight_layout(); plt.show()
    del sp_feats, sp_world
else:
    print("RUN_ABLATIONS is False: the spacing sweep was skipped")

In [ ]:
# --------------------------------------------------------------------------- #
# 23g. The weights and the biases.
# --------------------------------------------------------------------------- #
def parameter_kind(name, tensor):
    if name.endswith(".bias"):
        return "bias"
    if tensor.ndim == 1:
        return "norm scale (gamma)"
    return "conv / linear weight"


_wdf = pd.DataFrame([{
    "tensor": n, "kind": parameter_kind(n, p), "shape": "x".join(str(s) for s in p.shape),
    "count": p.numel(), "mean": float(p.detach().float().mean()),
    "std": float(p.detach().float().std()), "min": float(p.detach().float().min()),
    "max": float(p.detach().float().max()), "L2 norm": float(p.detach().float().norm())}
    for n, p in model.named_parameters()])
display(_wdf.round(4))

_split = _wdf.groupby("kind")["count"].agg(["count", "sum"])
_split["share %"] = _split["sum"] / _split["sum"].sum() * 100
print("parameters by kind ('count' is the number of tensors)")
display(_split.round(3))

_n_bn_shift = sum(p.numel() for n, p in model.named_parameters()
                  if n.endswith(".bias") and not n.startswith("head.4"))
_n_conv_bias = sum(p.numel() for n, p in model.named_parameters()
                   if n.endswith(".bias") and n.startswith("head.4"))
_n_gamma = sum(p.numel() for n, p in model.named_parameters()
               if n.endswith(".weight") and p.ndim == 1)
print(f"\nwhere the bias is: every convolution in the trunk is built with bias=False because a")
print(f"batch-norm layer follows it, and batch-norm subtracts the mean -- a bias there would")
print(f"cancel on every forward pass. So the only biases in the model are:")
print(f"  batch-norm shifts (beta)                 {_n_bn_shift:>7,}")
print(f"  the final 1x1 classification convolution {_n_conv_bias:>7,}  "
      f"(head.4, the one conv with no batch-norm after it)")
print(f"  batch-norm scales (gamma, also 1-D)      {_n_gamma:>7,}")
print(f"  total 1-D parameters                     "
      f"{_n_bn_shift + _n_conv_bias + _n_gamma:>7,} of {param_total:,} "
      f"({(_n_bn_shift + _n_conv_bias + _n_gamma) / param_total * 100:.2f}%)")

print("\nclass weights in the loss (inverse square-root frequency, normalised to mean 1):")
display(pd.DataFrame({"class": [f"{i}: {CLASS_NAMES[i]}" for i in range(NUM_CLASSES)],
                      "weight": CLASS_WEIGHTS.detach().cpu().numpy().round(4),
                      "train points": np.bincount(TR_LABELS.reshape(-1).cpu().numpy(),
                                                  minlength=NUM_CLASSES)}))

def _flat(pred):
    return np.concatenate([p.detach().cpu().float().reshape(-1).numpy()
                           for n, p in model.named_parameters() if pred(n, p)])


_conv_w = _flat(lambda n, p: not n.endswith(".bias") and p.ndim >= 2)
_gamma_v = _flat(lambda n, p: n.endswith(".weight") and p.ndim == 1)
_beta_v = _flat(lambda n, p: n.endswith(".bias") and not n.startswith("head.4"))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
axes[0].hist(_conv_w, bins=120, color="#4477aa")
axes[0].set_yscale("log")
axes[0].set_title(f"convolution / linear weights\n{_conv_w.size:,} values, mean "
                  f"{_conv_w.mean():+.4f}, std {_conv_w.std():.4f}", fontsize=11)
axes[0].set_xlabel("weight value")
axes[0].set_ylabel("count (log)")
axes[1].hist(_gamma_v, bins=60, alpha=0.75, label=f"gamma ({_gamma_v.size})", color="#228833")
axes[1].hist(_beta_v, bins=60, alpha=0.75, label=f"beta ({_beta_v.size})", color="#ee6677")
axes[1].set_yscale("log")
axes[1].set_title("batch-norm scale and shift\ninitialised at gamma=1, beta=0", fontsize=11)
axes[1].set_xlabel("value")
axes[1].legend(fontsize=9)
_head_w = model.head[4].weight.detach().cpu().float().numpy()[:, :, 0]
_v = float(np.abs(_head_w).max())
_im = axes[2].imshow(_head_w, cmap="coolwarm", aspect="auto", vmin=-_v, vmax=_v)
axes[2].set_yticks(range(NUM_CLASSES))
axes[2].set_yticklabels(CLASS_NAMES, fontsize=8)
axes[2].set_xlabel(f"last hidden channel (0..{_head_w.shape[1] - 1})")
axes[2].set_title("classification head weights\n8 x 64, the only conv with a bias", fontsize=11)
fig.colorbar(_im, ax=axes[2], fraction=0.046)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(NUM_CLASSES), CLASS_WEIGHTS.detach().cpu().numpy(), color="#4477aa")
axes[0].set_xticks(range(NUM_CLASSES))
axes[0].set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=9)
axes[0].set_ylabel("weight")
axes[0].grid(True, axis="y", alpha=0.3)
axes[0].set_title("class weights in the loss", fontsize=11)
_hb = model.head[4].bias.detach().cpu().float().numpy()
axes[1].bar(range(NUM_CLASSES), _hb, color="#ee6677")
axes[1].set_xticks(range(NUM_CLASSES))
axes[1].set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=9)
axes[1].set_ylabel("bias")
axes[1].grid(True, axis="y", alpha=0.3)
axes[1].set_title(f"the {NUM_CLASSES} head biases", fontsize=11)
plt.tight_layout(); plt.show()

## 24. The model on Birmingham, and what colour each class is

Section 21 pointed the model at `Paris_Luxembourg_6.ply`, a French mobile scan. This last cell
points it somewhere further from home again: **`Birmingham_3D/birmingham_block_0.ply`**, an
aerial photogrammetry block from **SensatUrban** (Hu et al., CVPR 2021) -- a different sensor,
a different country, a different viewing geometry (a UAV looking down, not a car looking along
a street), and a different label set of **13 classes** rather than the 8 this model knows.

Two things have to be said plainly before any number:

1. The model has **never seen aerial data**, so this is a domain-transfer test and not a
   benchmark. Nothing here is fine-tuned.
2. The label spaces only *partly* correspond. SensatUrban separates Ground, Road, Footpath,
   Parking and Rail; this model has one class for all five (`Road`). The mapping is a choice,
   printed below, not a fact -- which is why it is applied to the ground truth before scoring,
   so the comparison is like-for-like at 8 classes.

The prediction is therefore scored against the block's **own ground-truth labels** (SensatUrban
ships them; this is not something inferred here) after that mapping, and the numbers should be
read as a floor.

One knob is worth naming. The prediction runs on a 3M-point sample of the block's 22.0M, and
more points is not automatically better here: each inference block gathers the 1024 points
nearest its anchor, so a denser cloud gives every block a *smaller* footprint and leaves more
points outside every block. Measured on this block, raising the sample from 3M to 6M dropped
direct coverage from 98.3% to 85.7% and OA from 69.1% to 67.0%. The sample size below is the
one that scored best, not the one that merely fits in memory.

The closing figure is the legend the whole notebook has been relying on without ever stating:
which class index is which colour, for the model's 8 classes and for SensatUrban's 13, in the
palettes the renders above actually used.

In [ ]:
# --------------------------------------------------------------------------- #
# 24. Birmingham (SensatUrban): predict a block from a third dataset, and put
#     the class -> colour mapping on the record.
# --------------------------------------------------------------------------- #
from matplotlib.patches import Patch, Rectangle

BIRM_DIR = DATA_ROOT / "Birmingham_3D"
BIRM_TILE = BIRM_DIR / "birmingham_block_0.ply"
BIRM_MAX_POINTS = 3_000_000     # of the block's 22.0M points; None = every point
BIRM_SPACING = 2.0              # metres between block anchors, as used for the Toronto tiles
BIRM_RES = 700                  # pixels for the top-down panels: about 0.5 m per pixel, so a
                                #   majority vote is taken over the ~6 points in each cell

# SensatUrban's 13 classes, in the dataset's own order, with the colours these renders use.
SENSATURBAN_CLASSES = ["Ground", "Vegetation", "Building", "Wall", "Bridge", "Parking", "Rail",
                       "Road", "Street Furniture", "Car", "Footpath", "Bike", "Water"]
SENSATURBAN_PALETTE = np.array([
    [200, 180, 140],   # 0  Ground           - tan
    [ 60, 150,  60],   # 1  Vegetation       - green
    [210, 120, 100],   # 2  Building         - brick
    [110,  80,  60],   # 3  Wall             - brown
    [150, 100, 200],   # 4  Bridge           - purple
    [ 80,  85, 100],   # 5  Parking          - slate
    [ 45,  45,  55],   # 6  Rail             - near black
    [135, 140, 145],   # 7  Road             - grey
    [220, 190,  60],   # 8  Street Furniture - yellow
    [240,  90, 180],   # 9  Car              - pink
    [180, 180, 175],   # 10 Footpath         - light grey
    [ 40, 200, 210],   # 11 Bike             - cyan
    [ 55, 110, 220],   # 12 Water            - blue
], np.float32) / 255.0


# SensatUrban 13 -> this notebook's 8. Five of the thirteen collapse into `Road`, because the
# shared taxonomy has no way to tell a pavement from a car park from a rail bed.
SENSATURBAN_TO_SHARED = np.array([0, 2, 3, 7, 3, 0, 0, 0, 5, 6, 0, 6, 2], np.int64)


def load_sensaturban(path, max_points=None, seed=SEED, verbose=True):
    '''A SensatUrban block -> (world XYZ, model features, its own 13-class labels, RGB).

    SensatUrban's PLY is not the same file layout as the Toronto or Paris ones: 16 bytes per
    vertex as float32 XYZ + uint8 R,G,B + uint8 class, and the label field is called `class`.
    That class channel is what makes the block scorable at all -- it is the dataset's own
    ground truth, not something inferred here.
    '''
    ply = PlyData.read(str(path))
    v = ply["vertex"].data
    low = {f.lower(): f for f in v.dtype.names}
    for need in ("x", "y", "z", "class"):
        if need not in low:
            raise ValueError(f"missing {need} in {path}: {list(v.dtype.names)}")
    n = len(v)
    keep = (np.sort(np.random.default_rng(seed).choice(n, max_points, replace=False))
            if (max_points is not None and n > max_points) else np.arange(n))
    world = np.column_stack([v[low["x"]][keep], v[low["y"]][keep],
                             v[low["z"]][keep]]).astype(np.float64)
    labels = np.asarray(v[low["class"]][keep], np.int64)
    rgb = np.column_stack([v[low["red"]][keep], v[low["green"]][keep],
                           v[low["blue"]][keep]]).astype(np.float32) / 255.0
    feats = np.concatenate([world - world.mean(axis=0), rgb], axis=1).astype(np.float32)
    if verbose:
        print(f"  {path.name}: {n:,} points, fields {list(v.dtype.names)}"
              + (f", thinning to {len(keep):,}" if len(keep) < n else ""))
    del ply, v
    return world, feats, labels, rgb


def top_down(x, y, labels, resolution, palette=None, rgb=None, background=0.10):
    '''Majority-vote class image for any palette, or the per-cell mean colour when `rgb` is
    given. `render_class_map` is hard-wired to the 8-class palette and to NUM_CLASSES bins;
    this needs 13, so here is the general version. With no palette given it uses the notebook's
    8-class PALETTE.'''
    palette = PALETTE if palette is None else palette
    x = np.asarray(x, np.float64)
    y = np.asarray(y, np.float64)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    px = np.clip(((x - x.min()) * (resolution - 1) / max(x.max() - x.min(), 1e-9)).astype(np.int64),
                 0, resolution - 1)
    py = np.clip(((y - y.min()) * (resolution - 1) / max(y.max() - y.min(), 1e-9)).astype(np.int64),
                 0, resolution - 1)
    cell = py * resolution + px
    n = resolution * resolution
    if rgb is not None:
        c = np.asarray(rgb, np.float32)[ok].astype(np.float64)
        img = np.zeros((n, 3), np.float64)
        cnt = np.bincount(cell, minlength=n).astype(np.float64)
        for k in range(3):
            img[:, k] = np.bincount(cell, weights=c[:, k], minlength=n)
        nz = cnt > 0
        img[nz] /= cnt[nz, None]
        img[~nz] = background
        return np.flipud(img.reshape(resolution, resolution, 3)).clip(0, 1)
    K = len(palette)
    lab = np.asarray(labels, np.int64)[ok]
    good = (lab >= 0) & (lab < K)
    hist = np.bincount(cell[good] * K + lab[good], minlength=n * K).reshape(n, K)
    img = np.asarray(palette, np.float64)[hist.argmax(-1)].copy()
    img[hist.sum(-1) == 0] = background
    return np.flipud(img.reshape(resolution, resolution, 3))


if not BIRM_TILE.exists():
    print(f"{BIRM_TILE} not found -- put the SensatUrban block there and rerun this cell")
else:
    print(f"=== {BIRM_TILE.name} ({BIRM_TILE.stat().st_size / 1e6:.0f} MB) ===")
    t0 = time.time()
    birm_world, birm_feats, birm_gt, birm_rgb = load_sensaturban(BIRM_TILE,
                                                                max_points=BIRM_MAX_POINTS)
    _area = max(np.ptp(birm_world[:, 0]) * np.ptp(birm_world[:, 1]), 1e-9)
    print(f"  {len(birm_feats):,} points in {len(birm_feats) / _area:.0f} pts/m2 | "
          f"X {birm_world[:, 0].min():.0f}..{birm_world[:, 0].max():.0f} | "
          f"Y {birm_world[:, 1].min():.0f}..{birm_world[:, 1].max():.0f} | "
          f"Z {birm_world[:, 2].min():.1f}..{birm_world[:, 2].max():.1f} | "
          f"loaded in {time.time() - t0:.0f}s")

    t0 = time.time()
    birm_pred, birm_prob, birm_cov = predict_tile(birm_feats, model, spacing=BIRM_SPACING,
                                                  verbose=False)
    dt = time.time() - t0
    print(f"  predicted in {dt:.0f}s ({len(birm_pred) / dt / 1e3:.0f}k pts/s) | direct coverage "
          f"{birm_cov.mean() * 100:.1f}% | mean confidence {birm_prob.max(axis=1).mean():.3f}")

    _count = np.bincount(birm_gt, minlength=len(SENSATURBAN_CLASSES))
    print("\n  what is in the block (SensatUrban's own labels), and where each maps:")
    for i in np.argsort(-_count):
        if _count[i]:
            print(f"    {i:2d} {SENSATURBAN_CLASSES[i]:17}{_count[i]:>10,}  "
                  f"{_count[i] / len(birm_gt) * 100:5.1f}%  ->  "
                  f"{CLASS_NAMES[SENSATURBAN_TO_SHARED[i]]}")
    print("\n  what the model predicted there:")
    _share = np.bincount(birm_pred, minlength=NUM_CLASSES) / len(birm_pred)
    for i in np.argsort(-_share):
        if _share[i] >= 0.001:
            print(f"    {i} {CLASS_NAMES[i]:15}{_share[i] * 100:6.1f}%")

    birm_rep = class_report(SENSATURBAN_TO_SHARED[birm_gt], birm_pred)
    print_report(birm_rep, "Birmingham block_0 vs its own labels, mapped to the 8 shared classes")
    print("  read this as a floor: 8-class street-view model, 13-class aerial block, no")
    print("  fine-tuning, and a label mapping that merges five SensatUrban classes into Road.")
    np.save(OUTPUT_DIR / f"pred_BIRM_{BIRM_TILE.stem}.npy", birm_pred)
    print(f"  saved {OUTPUT_DIR / ('pred_BIRM_' + BIRM_TILE.stem + '.npy')}")

    # ---- what it looks like ------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(16.5, 8.4))
    axes[0].imshow(top_down(birm_world[:, 0], birm_world[:, 1], None, BIRM_RES, rgb=birm_rgb),
                   interpolation="nearest")
    axes[0].set_title(f"the block's own aerial RGB\n{len(birm_feats):,} points at "
                      f"{BIRM_RES}px ({len(birm_feats) / BIRM_RES ** 2:.1f} per cell)", fontsize=12)
    axes[1].imshow(top_down(birm_world[:, 0], birm_world[:, 1], birm_gt, BIRM_RES,
                            palette=SENSATURBAN_PALETTE), interpolation="nearest")
    axes[1].set_title("SensatUrban ground truth\n13 classes", fontsize=12)
    axes[2].imshow(render_class_map(birm_feats[:, 0], birm_feats[:, 1], birm_pred,
                                    resolution=BIRM_RES), interpolation="nearest")
    axes[2].set_title("this model's prediction\n8 shared classes, no fine-tuning", fontsize=12)
    for _a in axes:
        _a.set_xticks([])
        _a.set_yticks([])
    fig.legend(handles=[Patch(facecolor=PALETTE[i], label=f"{i}: {CLASS_NAMES[i]}")
                        for i in range(NUM_CLASSES)],
               loc="lower center", ncol=NUM_CLASSES, frameon=False, fontsize=10.5)
    plt.tight_layout(rect=(0, 0.035, 1, 1))
    plt.show()

    # The block is 143 x 344 m across but only 22 m tall, so a whole-block view is nearly a plan
    # view whatever the pitch. The second row crops to a 110 m window, re-framed by render_3d,
    # so the vertical structure -- roofs, gables, walls, canopies -- is actually visible and the
    # model's mistakes have somewhere to show themselves.
    _yaw = principal_yaw(birm_world, None)
    _cx = float(np.median(birm_world[:, 0]))
    _cy = float(np.median(birm_world[:, 1]))
    _half = 55.0
    _box = ((_cx - _half, _cy - _half), (_cx + _half, _cy + _half))
    _panels = [("predicted, whole block", birm_pred, PALETTE, NUM_CLASSES),
               ("SensatUrban ground truth, whole block", birm_gt, SENSATURBAN_PALETTE,
                len(SENSATURBAN_CLASSES)),
               (f"predicted, {2 * _half:.0f} m detail", birm_pred, PALETTE, NUM_CLASSES),
               ("ground truth, same detail", birm_gt, SENSATURBAN_PALETTE,
                len(SENSATURBAN_CLASSES))]
    fig, axes = plt.subplots(2, 2, figsize=(15.5, 12.4))
    for _ax, (_t, _lab, _pal, _k) in zip(axes.ravel(), _panels):
        _ax.imshow(render_3d(birm_world, _lab, yaw=_yaw, pitch=30, resolution=(1000, 760),
                             palette=_pal, num_classes=_k,
                             crop=None if "whole" in _t else _box))
        _ax.set_title(_t, fontsize=12)
        _ax.set_xticks([])
        _ax.set_yticks([])
    plt.tight_layout()
    plt.show()

    confusion_figure(birm_rep, "Birmingham: what each true (mapped) class was predicted as")
    plt.tight_layout()
    plt.show()

    # ---- the legend: class -> colour, for both taxonomies ------------------
    def _swatches(ax, rows, title):
        ax.axis("off")
        for k, (idx, name, colour, extra) in enumerate(rows):
            yy = len(rows) - k - 1
            ax.add_patch(Rectangle((0.0, yy + 0.10), 0.075, 0.80, facecolor=colour,
                                   edgecolor="0.35", linewidth=0.8))
            ax.text(0.095, yy + 0.5, f"{idx}: {name}", va="center", fontsize=11.5)
            ax.text(0.455, yy + 0.5,
                    "#" + "".join(f"{int(round(v * 255)):02X}" for v in colour),
                    va="center", fontsize=10.5, family="monospace")
            ax.text(0.635, yy + 0.5, extra, va="center", fontsize=10.5, color="0.25")
        ax.set_xlim(0, 1.0)
        ax.set_ylim(0, len(rows))
        ax.set_title(title, fontsize=12.5, pad=12)

    fig, axes = plt.subplots(1, 2, figsize=(17, 7.6))
    _swatches(axes[0],
              [(i, CLASS_NAMES[i], PALETTE[i],
                f"rgb({int(round(PALETTE[i][0] * 255))}, {int(round(PALETTE[i][1] * 255))}, "
                f"{int(round(PALETTE[i][2] * 255))})") for i in range(NUM_CLASSES)],
              "the 8 shared classes -- the palette every render above uses")
    _swatches(axes[1],
              [(i, SENSATURBAN_CLASSES[i], SENSATURBAN_PALETTE[i],
                (f"{_count[i] / len(birm_gt) * 100:4.1f}% of the block  ->  "
                 f"{CLASS_NAMES[SENSATURBAN_TO_SHARED[i]]}") if _count[i]
                else f"not in this block  ->  {CLASS_NAMES[SENSATURBAN_TO_SHARED[i]]}")
               for i in range(len(SENSATURBAN_CLASSES))],
              "SensatUrban's 13 classes -- what this block is labelled with")
    plt.tight_layout()
    plt.show()

    _photos = DATA_ROOT / "Birmingham_Mesh_Photos"
    if _photos.exists():
        print(f"\n  the same block also exists as a segmented triangle mesh with a rendered "
              f"photo set ({len(list(_photos.rglob('*.png')))} PNGs + turntables)"
              f" under {_photos}")